# TechNova Enterprise RAG Assistant — Clean Development Notebook

This notebook contains the retained project workflow only.

**Project:** Enterprise RAG Assistant for TechNova Solutions

**Core components:** Google Drive → documents → chunking → embeddings → FAISS → RAG engine → Streamlit UI → document-specific retrieval

**Important:** API keys must be supplied through Colab Secrets/environment variables. Never place a real API key directly in a notebook cell.

In [1]:
from google.colab import drive
import os
import shutil # Import shutil for recursive directory removal

# Define the mountpoint
MOUNTPOINT = '/content/drive'

# Check if the mountpoint exists and is a directory
# If it exists and is not empty, it could cause issues with drive.mount() even with force_remount=True.
# To ensure a clean state, we'll try to recursively remove it.
if os.path.isdir(MOUNTPOINT):
    if os.listdir(MOUNTPOINT):
        print(f"Warning: Mountpoint {MOUNTPOINT} exists and is not empty. Attempting to recursively remove its contents for a clean mount.")
        # Attempt to recursively remove the directory and its contents.
        try:
            shutil.rmtree(MOUNTPOINT) # Use shutil.rmtree for recursive deletion
            print(f"Successfully removed contents of {MOUNTPOINT}.")
        except OSError as e:
            print(f"Could not recursively remove {MOUNTPOINT}: {e}. Proceeding with mount attempt, but this may lead to further errors.")
    else:
        print(f"Mountpoint {MOUNTPOINT} exists but is empty. Proceeding with mount.")
elif os.path.exists(MOUNTPOINT):
    print(f"Warning: Mountpoint {MOUNTPOINT} exists but is not a directory. Removing it.")
    try:
        os.remove(MOUNTPOINT) # Remove if it's a file or symlink
        print(f"Successfully removed {MOUNTPOINT} (was not a directory).")
    except OSError as e:
        print(f"Could not remove {MOUNTPOINT} (was not a directory): {e}. Proceeding with mount attempt.")

# Mount Google Drive to access files, forcing a remount if already mounted or if the directory is not empty
# drive.mount will create the MOUNTPOINT directory if it doesn't exist after removal.
drive.mount(MOUNTPOINT, force_remount=True)

Mounted at /content/drive


In [2]:
# Define the main project directory within Google Drive
PROJECT_DIR = "/content/drive/MyDrive/Enterprise_RAG_Assistant"

# Print the project directory path
print(PROJECT_DIR)

/content/drive/MyDrive/Enterprise_RAG_Assistant


In [3]:
# List of subfolders expected within the project directory
folders = [
    "Data/documents",
    "Data/processed",
    "Notebooks",
    "src",
    "vectorstore",
    "outputs",
    "app"
]

# Iterate through each folder and check if it exists, then print the status
for folder in folders:
    path = os.path.join(PROJECT_DIR, folder)
    print(f"{folder:25} → {os.path.exists(path)}")

Data/documents            → True
Data/processed            → True
Notebooks                 → False
src                       → True
vectorstore               → True
outputs                   → False
app                       → True


In [4]:
# Install necessary Python packages silently
# pypdf: for PDF document processing
# python-docx: for DOCX document processing
# sentence-transformers: for generating text embeddings
# faiss-cpu: for efficient similarity search and clustering of dense vectors
# chromadb: a vector database
!pip install -q pypdf python-docx sentence-transformers faiss-cpu chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95

In [5]:
from docx import Document
from pathlib import Path

# ============================================================
# STEP 3A — CREATE TECHNOVA ORGANIZATION DOCUMENTS
# ============================================================

# Define the directory where the DOCX documents will be saved
DOCUMENTS_DIR = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/Data/documents"
)

# Create the directory if it doesn't already exist
DOCUMENTS_DIR.mkdir(parents=True, exist_ok=True)


def create_document(filename, title, sections):
    """
    Create a DOCX document with a title and structured sections.

    Args:
        filename (str): The name of the DOCX file to be created.
        title (str): The main title of the document.
        sections (list): A list of tuples, where each tuple contains
                         a section heading and its content (string or list of strings).
    """
    doc = Document()

    # Add the main title to the document
    doc.add_heading(title, level=0)

    # Add organizational information
    doc.add_paragraph("TechNova Solutions")
    doc.add_paragraph("Internal Organizational Knowledge Document")

    # Iterate through the provided sections and add them to the document
    for heading, content in sections:
        doc.add_heading(heading, level=1)

        # If content is a list, add them as bullet points
        if isinstance(content, list):
            for item in content:
                doc.add_paragraph(item, style="List Bullet")
        # Otherwise, add as a regular paragraph
        else:
            doc.add_paragraph(content)

    # Define the full path for saving the document
    path = DOCUMENTS_DIR / filename
    doc.save(path)

    return path


# ============================================================
# 1. EMPLOYEE HANDBOOK
# ============================================================

# Create the Employee Handbook document
create_document(
    "Employee_Handbook.docx",
    "Employee Handbook",
    [
        (
            "1. About TechNova Solutions",
            "TechNova Solutions is a technology organization focused on software "
            "engineering, artificial intelligence, cloud computing, and digital "
            "business solutions."
        ),
        (
            "2. Working Hours",
            "The standard working schedule is Monday through Friday. "
            "Employees are generally expected to work eight hours per working day. "
            "Core collaboration hours are from 10:00 AM to 4:00 PM."
        ),
        (
            "3. Employee Responsibilities",
            [
                "Employees must perform assigned responsibilities professionally.",
                "Employees must follow organizational policies and procedures.",
                "Employees must protect confidential company information.",
                "Employees must maintain respectful communication with colleagues.",
                "Employees must report security or compliance concerns promptly."
            ]
        ),
        (
            "4. Workplace Conduct",
            "Employees are expected to maintain a professional and respectful "
            "work environment and comply with the organization's Code of Conduct."
        ),
        (
            "5. Performance Reviews",
            "Employee performance is reviewed periodically based on responsibilities, "
            "goals, collaboration, quality of work, and professional development."
        )
    ]
)


# ============================================================
# 2. LEAVE POLICY
# ============================================================

# Create the Leave Policy document
create_document(
    "Leave_Policy.docx",
    "Leave Policy",
    [
        (
            "1. Annual Leave",
            "Full-time employees are entitled to 20 days of annual leave per "
            "calendar year. Annual leave should normally be requested in advance "
            "through the organization's leave management system."
        ),
        (
            "2. Sick Leave",
            "Employees may request sick leave when they are unable to work because "
            "of illness. Employees should notify their manager as soon as reasonably possible."
        ),
        (
            "3. Leave Approval",
            "Leave requests are subject to manager approval and operational requirements. "
            "Employees should avoid submitting leave requests at the last minute except "
            "in emergency situations."
        ),
        (
            "4. Carry Forward",
            "Up to 5 unused annual leave days may be carried forward to the following "
            "calendar year, subject to organizational rules."
        ),
        (
            "5. Emergency Leave",
            "Employees may request emergency leave for unexpected personal circumstances. "
            "The employee should inform the manager as soon as possible."
        )
    ]
)


# ============================================================
# 3. WORK FROM HOME POLICY
# ============================================================

# Create the Work From Home Policy document
create_document(
    "Work_From_Home_Policy.docx",
    "Work From Home Policy",
    [
        (
            "1. Eligibility",
            "Employees whose roles can be performed remotely may request work-from-home "
            "arrangements, subject to manager approval."
        ),
        (
            "2. Maximum Consecutive Remote Days",
            "Employees may work remotely for up to 3 consecutive working days under "
            "the standard work-from-home arrangement."
        ),
        (
            "3. Manager Approval",
            "Work-from-home requests should be submitted to the employee's manager "
            "before the planned remote-working period."
        ),
        (
            "4. Availability",
            "Employees working remotely must remain reachable during core collaboration "
            "hours from 10:00 AM to 4:00 PM."
        ),
        (
            "5. Security Requirements",
            "Employees working remotely must use approved devices, secure network "
            "connections, and follow all IT security requirements."
        )
    ]
)


# ============================================================
# 4. ATTENDANCE POLICY
# ============================================================

# Create the Attendance Policy document
create_document(
    "Attendance_Policy.docx",
    "Attendance Policy",
    [
        (
            "1. Standard Schedule",
            "The standard working schedule is Monday through Friday with eight "
            "working hours per day."
        ),
        (
            "2. Core Hours",
            "Core collaboration hours are from 10:00 AM to 4:00 PM. Employees "
            "are expected to be available during these hours unless they are on approved leave."
        ),
        (
            "3. Attendance Recording",
            "Employees must accurately record attendance using the organization's "
            "approved attendance system."
        ),
        (
            "4. Late Arrival",
            "Employees who expect to arrive late should notify their manager as soon "
            "as reasonably possible."
        ),
        (
            "5. Absence",
            "Employees who cannot attend work should inform their manager and follow "
            "the applicable leave procedure."
        )
    ]
)


# ============================================================
# 5. IT SECURITY POLICY
# ============================================================

# Create the IT Security Policy document
create_document(
    "IT_Security_Policy.docx",
    "IT Security Policy",
    [
        (
            "1. Password Requirements",
            "Passwords must contain at least 12 characters and should include a "
            "combination of uppercase letters, lowercase letters, numbers, and symbols."
        ),
        (
            "2. Password Protection",
            "Employees must not share passwords with other people. Passwords should "
            "not be written in publicly accessible locations."
        ),
        (
            "3. Multi-Factor Authentication",
            "Multi-factor authentication must be enabled for systems that require it "
            "under the organization's security standards."
        ),
        (
            "4. Device Security",
            "Employees must use approved security controls on company devices and "
            "should lock their devices when leaving them unattended."
        ),
        (
            "5. Security Incidents",
            "Suspected phishing, malware, unauthorized access, or loss of company "
            "devices must be reported to the IT security team promptly."
        )
    ]
)


# ============================================================
# 6. ACCEPTABLE USE POLICY
# ============================================================

# Create the Acceptable Use Policy document
create_document(
    "Acceptable_Use_Policy.docx",
    "Acceptable Use Policy",
    [
        (
            "1. Appropriate Use",
            "Company systems and network resources should primarily be used for "
            "authorized business activities."
        ),
        (
            "2. Prohibited Activities",
            [
                "Unauthorized access to company systems.",
                "Attempting to bypass security controls.",
                "Installing unauthorized software on company devices.",
                "Using company systems for illegal activities.",
                "Sharing confidential information with unauthorized parties."
            ]
        ),
        (
            "3. Software Installation",
            "Employees should obtain appropriate approval before installing software "
            "that is not provided or approved by the organization."
        ),
        (
            "4. Internet Usage",
            "Internet resources must be used responsibly and in accordance with "
            "company security and professional conduct requirements."
        ),
        (
            "5. Monitoring",
            "Company systems may be monitored in accordance with applicable organizational "
            "policies and legal requirements."
        )
    ]
)


# ============================================================
# 7. TRAVEL POLICY
# ============================================================

# Create the Business Travel Policy document
create_document(
    "Travel_Policy.docx",
    "Business Travel Policy",
    [
        (
            "1. Travel Authorization",
            "Business travel must receive appropriate authorization before bookings "
            "are made, except where emergency business circumstances require otherwise."
        ),
        (
            "2. Travel Booking",
            "Employees should use the organization's approved travel booking process "
            "for flights, accommodation, and other eligible travel arrangements."
        ),
        (
            "3. Transportation",
            "Reasonable transportation expenses incurred for approved business travel "
            "may be reimbursed according to organizational guidelines."
        ),
        (
            "4. Accommodation",
            "Employees should select reasonably priced accommodation that meets "
            "business requirements and organizational reimbursement limits."
        ),
        (
            "5. Expense Claims",
            "Employees must submit eligible travel expenses with appropriate receipts "
            "and supporting documentation after completing the trip."
        )
    ]
)


# ============================================================
# 8. PROCUREMENT SOP
# ============================================================

# Create the Procurement Standard Operating Procedure document
create_document(
    "Procurement_SOP.docx",
    "Procurement Standard Operating Procedure",
    [
        (
            "1. Procurement Request",
            "Employees requiring products or services should submit a procurement "
            "request describing the business requirement."
        ),
        (
            "2. Manager Approval",
            "Procurement requests require appropriate manager approval before the "
            "purchasing process begins."
        ),
        (
            "3. Vendor Selection",
            "Procurement personnel should evaluate suitable vendors based on factors "
            "such as price, quality, delivery requirements, and organizational needs."
        ),
        (
            "4. Purchase Approval",
            "Purchases must receive the required authorization before an order is placed."
        ),
        (
            "5. Documentation",
            "Procurement records should include the request, approval information, "
            "vendor details, purchase documentation, and relevant invoices."
        )
    ]
)


# ============================================================
# VERIFY FILES
# ============================================================

print("✅ Documents created successfully!\n")

# List all DOCX files in the documents directory
files = sorted(DOCUMENTS_DIR.glob("*.docx"))

# Print the name of each created document
for i, file in enumerate(files, start=1):
    print(f"{i}. {file.name}")

# Print the total number of documents and their storage location
print(f"\nTotal documents: {len(files)}")
print(f"Location: {DOCUMENTS_DIR}")

✅ Documents created successfully!

1. Acceptable_Use_Policy.docx
2. Attendance_Policy.docx
3. Employee_Handbook.docx
4. IT_Security_Policy.docx
5. Leave_Policy.docx
6. Procurement_SOP.docx
7. Travel_Policy.docx
8. Work_From_Home_Policy.docx

Total documents: 8
Location: /content/drive/MyDrive/Enterprise_RAG_Assistant/Data/documents


In [6]:
from pathlib import Path

# Define the directory where the DOCX documents are stored
DOCUMENTS_DIR = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/Data/documents"
)

# Find all DOCX files in the specified directory and sort them
files = sorted(DOCUMENTS_DIR.glob("*.docx"))

# Print the number of documents found
print("Documents found:", len(files))
print()

# Print the name of each found document
for i, file in enumerate(files, start=1):
    print(f"{i}. {file.name}")

Documents found: 8

1. Acceptable_Use_Policy.docx
2. Attendance_Policy.docx
3. Employee_Handbook.docx
4. IT_Security_Policy.docx
5. Leave_Policy.docx
6. Procurement_SOP.docx
7. Travel_Policy.docx
8. Work_From_Home_Policy.docx


In [7]:
from docx import Document
from pathlib import Path

# Define the directory containing the DOCX documents
DOCUMENTS_DIR = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/Data/documents"
)

# Select a specific document (Employee Handbook) for demonstration
file_path = DOCUMENTS_DIR / "Employee_Handbook.docx"

# Load the selected document
doc = Document(file_path)

# Print the document name and the total number of paragraphs it contains
print("Document:", file_path.name)
print("Number of paragraphs:", len(doc.paragraphs))

print("\n--- EXTRACTED TEXT ---\n")

# Iterate through each paragraph in the document
for paragraph in doc.paragraphs:
    # Get the text content of the paragraph and remove leading/trailing whitespace
    text = paragraph.text.strip()

    # If the paragraph is not empty, print its text
    if text:
        print(text)

Document: Employee_Handbook.docx
Number of paragraphs: 17

--- EXTRACTED TEXT ---

Employee Handbook
TechNova Solutions
Internal Organizational Knowledge Document
1. About TechNova Solutions
TechNova Solutions is a technology organization focused on software engineering, artificial intelligence, cloud computing, and digital business solutions.
2. Working Hours
The standard working schedule is Monday through Friday. Employees are generally expected to work eight hours per working day. Core collaboration hours are from 10:00 AM to 4:00 PM.
3. Employee Responsibilities
Employees must perform assigned responsibilities professionally.
Employees must follow organizational policies and procedures.
Employees must protect confidential company information.
Employees must maintain respectful communication with colleagues.
Employees must report security or compliance concerns promptly.
4. Workplace Conduct
Employees are expected to maintain a professional and respectful work environment and comply

In [11]:
from docx import Document
from pathlib import Path

# ============================================================
# STEP 4B — DOCUMENT LOADER
# ============================================================

# Define the directory containing the company DOCX documents
DOCUMENTS_DIR = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/Data/documents"
)


def load_docx_documents(directory):
    """
    Load all DOCX documents from the specified directory.

    Each paragraph becomes one document record with metadata.
    """

    documents = []

    # Find all DOCX files
    files = sorted(directory.glob("*.docx"))

    # Read every DOCX file
    for file_path in files:

        doc = Document(file_path)

        # Extract every non-empty paragraph
        for paragraph_id, paragraph in enumerate(doc.paragraphs):

            text = paragraph.text.strip()

            if not text:
                continue

            documents.append({
                "text": text,
                "document": file_path.name,
                "source": str(file_path),
                "paragraph_id": paragraph_id
            })

    return documents


# ============================================================
# LOAD ALL DOCUMENTS
# ============================================================

documents = load_docx_documents(DOCUMENTS_DIR)

print("✅ Documents loaded successfully!")
print("Total DOCX files:", len(list(DOCUMENTS_DIR.glob("*.docx"))))
print("Total text records:", len(documents))

✅ Documents loaded successfully!
Total DOCX files: 8
Total text records: 112


In [12]:
import re

# ============================================================
# STEP 5A — TEXT CLEANING
# ============================================================

def clean_text(text):
    """
    Clean extracted document text while preserving its meaning.
    Specifically, it converts multiple whitespace characters into a single space
    and removes leading/trailing whitespace.

    Args:
        text (str): The input text to be cleaned.

    Returns:
        str: The cleaned text.
    """

    # Convert multiple whitespace characters (spaces, tabs, newlines) into a single space
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing whitespace from the text
    text = text.strip()

    return text


# Initialize an empty list to store the cleaned document records
cleaned_documents = []

# Iterate through each original document record
for item in documents:

    # Clean the 'text' field of the current record
    cleaned = clean_text(item["text"])

    # If the text becomes empty after cleaning, skip this record
    if not cleaned:
        continue

    # Append the cleaned record with its original metadata to the new list
    cleaned_documents.append({
        "text": cleaned,
        "document": item["document"],
        "source": item["source"],
        "paragraph_id": item["paragraph_id"]
    })


# Print a confirmation message that text cleaning is complete
print("✅ Text cleaning completed!")
# Print the number of records before cleaning for comparison
print("Records before cleaning:", len(documents))
# Print the number of records after cleaning
print("Records after cleaning:", len(cleaned_documents))

✅ Text cleaning completed!
Records before cleaning: 112
Records after cleaning: 112


In [13]:
# ============================================================
# STEP 6A — CHUNKING + METADATA
# ============================================================

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100


def create_chunks(records, chunk_size=500, chunk_overlap=100):
    """
    Combine cleaned paragraph records into chunks.

    Chunks never cross document boundaries.
    Metadata from the original paragraphs is preserved.
    """

    chunks = []

    # Group records by document
    documents_grouped = {}

    for record in records:
        document_name = record["document"]

        if document_name not in documents_grouped:
            documents_grouped[document_name] = []

        documents_grouped[document_name].append(record)

    # Process each document separately
    for document_name, records_for_document in documents_grouped.items():

        current_text = ""
        current_paragraph_ids = []

        for record in records_for_document:

            paragraph_text = record["text"]

            # If adding the paragraph exceeds chunk size,
            # save the current chunk first.
            if (
                current_text
                and len(current_text) + len(paragraph_text) + 1 > chunk_size
            ):

                chunk_id = f"{document_name}_chunk_{len(chunks):04d}"

                chunks.append({
                    "chunk_id": chunk_id,
                    "text": current_text.strip(),
                    "document": document_name,
                    "source": record["source"],
                    "paragraph_ids": current_paragraph_ids.copy()
                })

                # Keep overlap from the end of the previous chunk
                overlap_text = current_text[-chunk_overlap:]

                current_text = overlap_text + " " + paragraph_text
                current_paragraph_ids = [record["paragraph_id"]]

            else:

                if current_text:
                    current_text += " " + paragraph_text
                else:
                    current_text = paragraph_text

                current_paragraph_ids.append(record["paragraph_id"])

        # Save final chunk for this document
        if current_text.strip():

            chunk_id = f"{document_name}_chunk_{len(chunks):04d}"

            chunks.append({
                "chunk_id": chunk_id,
                "text": current_text.strip(),
                "document": document_name,
                "source": records_for_document[-1]["source"],
                "paragraph_ids": current_paragraph_ids.copy()
            })

    return chunks


# Create chunks
chunks = create_chunks(
    cleaned_documents,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

print("✅ Chunking completed!")
print("Total cleaned records:", len(cleaned_documents))
print("Total chunks:", len(chunks))

✅ Chunking completed!
Total cleaned records: 112
Total chunks: 20


In [14]:
# ============================================================
# STEP 6C — IMPROVED WORD-AWARE CHUNKING
# ============================================================

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100


def create_chunks(records, chunk_size=500, chunk_overlap=100):
    """
    Create word-aware chunks while preserving document boundaries
    and metadata.
    """

    chunks = []

    # Group records by document
    documents_grouped = {}

    for record in records:
        documents_grouped.setdefault(
            record["document"], []
        ).append(record)

    # Process each document separately
    for document_name, doc_records in documents_grouped.items():

        current_words = []
        current_paragraph_ids = []

        for record in doc_records:

            paragraph_words = record["text"].split()

            # Add paragraph words
            for word in paragraph_words:

                current_words.append(word)

                # Check chunk size
                current_text = " ".join(current_words)

                if len(current_text) >= chunk_size:

                    chunk_number = len(chunks)

                    chunks.append({
                        "chunk_id": (
                            f"{document_name}_chunk_{chunk_number:04d}"
                        ),
                        "text": current_text,
                        "document": document_name,
                        "source": record["source"],
                        "paragraph_ids": current_paragraph_ids.copy()
                        if current_paragraph_ids
                        else [record["paragraph_id"]]
                    })

                    # Create word-based overlap
                    overlap_words = current_words[-20:]

                    current_words = overlap_words.copy()

                    current_paragraph_ids = [
                        record["paragraph_id"]
                    ]

            # Track paragraph
            if record["paragraph_id"] not in current_paragraph_ids:
                current_paragraph_ids.append(
                    record["paragraph_id"]
                )

        # Save remaining text
        if current_words:

            chunk_number = len(chunks)

            chunks.append({
                "chunk_id": (
                    f"{document_name}_chunk_{chunk_number:04d}"
                ),
                "text": " ".join(current_words),
                "document": document_name,
                "source": doc_records[-1]["source"],
                "paragraph_ids": current_paragraph_ids.copy()
            })

    return chunks


# Re-create chunks
chunks = create_chunks(
    cleaned_documents,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

print("✅ Improved chunking completed!")
print("Total cleaned records:", len(cleaned_documents))
print("Total chunks:", len(chunks))

✅ Improved chunking completed!
Total cleaned records: 112
Total chunks: 20


In [15]:
# ============================================================
# STEP 7B — LOAD EMBEDDING MODEL
# ============================================================

from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"

print("Loading embedding model...")

embedding_model = SentenceTransformer(MODEL_NAME)

print("✅ Embedding model loaded successfully!")
print("Model:", MODEL_NAME)

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded successfully!
Model: all-MiniLM-L6-v2


In [16]:
# ============================================================
# STEP 7C — GENERATE EMBEDDINGS
# ============================================================

# Extract chunk text
chunk_texts = [chunk["text"] for chunk in chunks]

print("Generating embeddings...")
print("Number of chunks:", len(chunk_texts))

# Generate embeddings
embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\n✅ Embeddings generated successfully!")
print("Embedding shape:", embeddings.shape)

Generating embeddings...
Number of chunks: 20


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Embeddings generated successfully!
Embedding shape: (20, 384)


In [17]:
# ============================================================
# STEP 8A — CREATE FAISS VECTOR DATABASE
# ============================================================

import faiss
import numpy as np

# Get embedding dimension
embedding_dimension = embeddings.shape[1]

print("Embedding dimension:", embedding_dimension)

# Create FAISS index
index = faiss.IndexFlatIP(embedding_dimension)

# Add embeddings to FAISS
index.add(embeddings.astype("float32"))

print("✅ FAISS vector database created successfully!")
print("Total vectors stored:", index.ntotal)

Embedding dimension: 384
✅ FAISS vector database created successfully!
Total vectors stored: 20


In [18]:
# ============================================================
# STEP 8B — SEMANTIC RETRIEVAL TEST
# ============================================================

query = "What are the password requirements?"

print("Question:", query)
print("\nGenerating question embedding...")

# Convert the question into the same vector space
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# Search FAISS
top_k = 3

scores, indices = index.search(
    query_embedding,
    top_k
)

print("\n✅ Retrieval completed!")
print("\nTop retrieved chunks:\n")

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):

    chunk = chunks[idx]

    print("=" * 70)
    print(f"Rank: {rank}")
    print(f"Similarity Score: {score:.4f}")
    print(f"Document: {chunk['document']}")
    print(f"Chunk ID: {chunk['chunk_id']}")
    print(f"Text: {chunk['text']}")
    print()

Question: What are the password requirements?

Generating question embedding...

✅ Retrieval completed!

Top retrieved chunks:

Rank: 1
Similarity Score: 0.6464
Document: IT_Security_Policy.docx
Chunk ID: IT_Security_Policy.docx_chunk_0008
Text: IT Security Policy TechNova Solutions Internal Organizational Knowledge Document 1. Password Requirements Passwords must contain at least 12 characters and should include a combination of uppercase letters, lowercase letters, numbers, and symbols. 2. Password Protection Employees must not share passwords with other people. Passwords should not be written in publicly accessible locations. 3. Multi-Factor Authentication Multi-factor authentication must be enabled for systems that require it under the

Rank: 2
Similarity Score: 0.4735
Document: IT_Security_Policy.docx
Chunk ID: IT_Security_Policy.docx_chunk_0009
Text: written in publicly accessible locations. 3. Multi-Factor Authentication Multi-factor authentication must be enabled for systems th

In [19]:
# ============================================================
# STEP 8C — REUSABLE SEMANTIC RETRIEVAL FUNCTION
# ============================================================

def retrieve(query, top_k=3):
    """
    Retrieve the most relevant chunks for a user question.
    """

    # Create embedding for the question
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Search FAISS
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        chunk = chunks[idx]

        results.append({
            "score": float(score),
            "chunk_id": chunk["chunk_id"],
            "document": chunk["document"],
            "text": chunk["text"],
            "paragraph_ids": chunk["paragraph_ids"],
            "source": chunk["source"]
        })

    return results


print("✅ Retrieval function created!")

✅ Retrieval function created!


In [20]:
# ============================================================
# STEP 8D — TEST RETRIEVAL WITH MULTIPLE QUESTIONS
# ============================================================

test_questions = [
    "How many annual leave days do employees get?",
    "How many consecutive days can I work from home?",
    "What are the password requirements?"
]

for question in test_questions:

    print("\n" + "=" * 80)
    print("QUESTION:", question)
    print("=" * 80)

    results = retrieve(question, top_k=2)

    for rank, result in enumerate(results, start=1):

        print(f"\nRank {rank}")
        print("Score:", round(result["score"], 4))
        print("Document:", result["document"])
        print("Chunk:", result["chunk_id"])
        print("Text:", result["text"][:500])


QUESTION: How many annual leave days do employees get?

Rank 1
Score: 0.6309
Document: Leave_Policy.docx
Chunk: Leave_Policy.docx_chunk_0010
Text: Leave Policy TechNova Solutions Internal Organizational Knowledge Document 1. Annual Leave Full-time employees are entitled to 20 days of annual leave per calendar year. Annual leave should normally be requested in advance through the organization's leave management system. 2. Sick Leave Employees may request sick leave when they are unable to work because of illness. Employees should notify their manager as soon as reasonably possible. 3. Leave Approval Leave requests are subject to manager app

Rank 2
Score: 0.4454
Document: Leave_Policy.docx
Chunk: Leave_Policy.docx_chunk_0011
Text: Employees should notify their manager as soon as reasonably possible. 3. Leave Approval Leave requests are subject to manager approval and operational requirements. Employees should avoid submitting leave requests at the last minute except in emergency situat

In [21]:
%pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 15.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [22]:
# ============================================================
# STEP 10B-19B — CONNECT COLAB TO PROJECT SOURCE DIRECTORY
# ============================================================

import sys
from pathlib import Path

# ------------------------------------------------------------
# Project directory
# ------------------------------------------------------------
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant"
)

# Source directory containing rag_engine.py
SRC_DIR = PROJECT_DIR / "src"

print("Project directory:")
print(PROJECT_DIR)

print("\nSource directory:")
print(SRC_DIR)

# ------------------------------------------------------------
# Check that the directories actually exist
# ------------------------------------------------------------
if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"❌ Project directory not found: {PROJECT_DIR}"
    )

if not SRC_DIR.exists():
    raise FileNotFoundError(
        f"❌ Source directory not found: {SRC_DIR}"
    )

rag_file = SRC_DIR / "rag_engine.py"

if not rag_file.exists():
    raise FileNotFoundError(
        f"❌ rag_engine.py not found: {rag_file}"
    )

print("\n✅ Project directory found.")
print("✅ Source directory found.")
print("✅ rag_engine.py found.")

# ------------------------------------------------------------
# Add BOTH project and src directories to Python path
# ------------------------------------------------------------
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("\n✅ Python import paths configured.")

Project directory:
/content/drive/MyDrive/Enterprise_RAG_Assistant

Source directory:
/content/drive/MyDrive/Enterprise_RAG_Assistant/src

✅ Project directory found.
✅ Source directory found.
✅ rag_engine.py found.

✅ Python import paths configured.


In [23]:
# ============================================================
# STEP 6.4 — UPDATE THE REAL RAG ENGINE
# ============================================================

from pathlib import Path

# Path to the existing RAG engine
RAG_ENGINE_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/src/rag_engine.py"
)

# ------------------------------------------------------------
# Updated RAG engine
# ------------------------------------------------------------

rag_engine_code = r'''
# ============================================================
# TECHNOVA ENTERPRISE RAG ENGINE
# ============================================================

from pathlib import Path
import pickle
import time

import faiss
from sentence_transformers import SentenceTransformer
from google import genai


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(__file__).resolve().parent.parent

VECTORSTORE_DIR = PROJECT_DIR / "vectorstore"

FAISS_PATH = VECTORSTORE_DIR / "tech_nova.index"
CHUNKS_PATH = VECTORSTORE_DIR / "chunks.pkl"


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "all-MiniLM-L6-v2"

GEMINI_MODEL = "gemini-3.6-flash"

# Minimum FAISS similarity score required for a result
# to be considered relevant.
RELEVANCE_THRESHOLD = 0.20


# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

print("Loading embedding model...")

embedding_model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded:", MODEL_NAME)


# ============================================================
# LOAD FAISS INDEX
# ============================================================

print("Loading FAISS index...")

index = faiss.read_index(str(FAISS_PATH))

print("FAISS index loaded!")
print("Vectors:", index.ntotal)
print("Dimension:", index.d)


# ============================================================
# LOAD CHUNKS
# ============================================================

print("Loading chunks...")

with open(CHUNKS_PATH, "rb") as f:
    chunks = pickle.load(f)

print("Chunks loaded:", len(chunks))


# ============================================================
# GEMINI CLIENT
# ============================================================

client = genai.Client()


# ============================================================
# RETRIEVAL
# ============================================================

def retrieve(query, top_k=3):
    """
    Retrieve the most relevant document chunks for a query.

    Only chunks with a similarity score greater than or equal
    to RELEVANCE_THRESHOLD are returned.
    """

    # Convert the user's question into an embedding.
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Search the FAISS vector database.
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    # Process each retrieved result.
    for score, idx in zip(scores[0], indices[0]):

        # FAISS can return -1 when no valid result exists.
        if idx < 0:
            continue

        # Convert the score to a normal Python float.
        score = float(score)

        # ----------------------------------------------------
        # RELEVANCE FILTER
        # ----------------------------------------------------
        # Ignore documents below our minimum relevance score.
        if score < RELEVANCE_THRESHOLD:
            continue

        # Get the corresponding chunk.
        chunk = chunks[idx]

        # Store the useful metadata.
        results.append({
            "document": chunk["document"],
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "score": score
        })

    return results


# ============================================================
# RAG PROMPT
# ============================================================

def build_rag_prompt(question, retrieved_results):

    context_parts = []

    for i, result in enumerate(
        retrieved_results,
        start=1
    ):

        context_parts.append(
            f"""
SOURCE {i}
Document: {result["document"]}
Chunk ID: {result["chunk_id"]}

Content:
{result["text"]}
"""
        )

    context = "\n".join(context_parts)

    prompt = f"""
You are TechNova Solutions' internal knowledge assistant.

Answer the user's question using ONLY the information
provided in the retrieved company documents below.

Rules:
1. Do not invent information.
2. Do not use outside knowledge.
3. If the retrieved documents do not contain enough information,
   say that the information is not available in the provided
   company documents.
4. Give a concise and clear answer.
5. Mention the relevant document name in the answer when useful.

RETRIEVED COMPANY DOCUMENTS:

{context}

USER QUESTION:
{question}

ANSWER:
"""

    return prompt


# ============================================================
# GEMINI RETRY
# ============================================================

def generate_with_retry(prompt, max_retries=4):
    """
    Generate a Gemini response with automatic retry handling.

    If Gemini temporarily returns a server error such as 503,
    the function waits and tries again.
    """

    for attempt in range(max_retries):

        try:

            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt
            )

            return response.text.strip()

        except Exception as e:

            # Retry if attempts remain.
            if attempt < max_retries - 1:

                wait_time = 2 ** attempt

                print(
                    f"Gemini request failed. "
                    f"Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:

                # All retries failed.
                raise e


# ============================================================
# COMPLETE RAG PIPELINE
# ============================================================

def answer_question(question, top_k=3):
    """
    Complete RAG pipeline:

    1. Retrieve relevant chunks.
    2. Apply relevance threshold.
    3. If nothing relevant is found, return a safe response.
    4. Otherwise send the retrieved context to Gemini.
    5. Return answer and sources.
    """

    # --------------------------------------------------------
    # STEP 1: RETRIEVE RELEVANT DOCUMENTS
    # --------------------------------------------------------

    retrieved_results = retrieve(
        question,
        top_k=top_k
    )

    # --------------------------------------------------------
    # STEP 2: HANDLE NO RELEVANT DOCUMENTS
    # --------------------------------------------------------

    if not retrieved_results:

        return {
            "question": question,

            "answer": (
                "This information is not available in "
                "the provided company documents."
            ),

            "sources": []
        }

    # --------------------------------------------------------
    # STEP 3: BUILD GROUNDED RAG PROMPT
    # --------------------------------------------------------

    prompt = build_rag_prompt(
        question,
        retrieved_results
    )

    # --------------------------------------------------------
    # STEP 4: GENERATE ANSWER
    # --------------------------------------------------------

    answer = generate_with_retry(
        prompt
    )

    # --------------------------------------------------------
    # STEP 5: PREPARE SOURCES
    # --------------------------------------------------------

    sources = []

    for result in retrieved_results:

        sources.append({
            "document": result["document"],
            "chunk_id": result["chunk_id"],
            "score": result["score"]
        })

    # --------------------------------------------------------
    # STEP 6: RETURN FINAL RESULT
    # --------------------------------------------------------

    return {
        "question": question,
        "answer": answer,
        "sources": sources
    }
'''

# Write the updated code to rag_engine.py
RAG_ENGINE_PATH.write_text(
    rag_engine_code,
    encoding="utf-8"
)

print("✅ RAG engine updated successfully!")
print("Location:", RAG_ENGINE_PATH)
print("Relevance threshold: 0.20")

✅ RAG engine updated successfully!
Location: /content/drive/MyDrive/Enterprise_RAG_Assistant/src/rag_engine.py
Relevance threshold: 0.20


In [24]:
# ============================================================
# STEP 10B-21 — UPDATE SMART GEMINI RETRY LOGIC
# ============================================================

from pathlib import Path

RAG_ENGINE_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/src/rag_engine.py"
)

# Read current RAG engine
code = RAG_ENGINE_PATH.read_text(encoding="utf-8")

# ------------------------------------------------------------
# Add the required Google GenAI error imports
# ------------------------------------------------------------

old_import = "from google import genai"

new_import = """from google import genai
from google.genai.errors import ClientError, ServerError"""

if old_import in code and "from google.genai.errors import" not in code:
    code = code.replace(old_import, new_import)


# ------------------------------------------------------------
# Replace generate_with_retry() with smarter retry logic
# ------------------------------------------------------------

start_marker = "# ============================================================\n# GEMINI RETRY\n# ============================================================"

end_marker = "# ============================================================\n# COMPLETE RAG PIPELINE\n# ============================================================"

start = code.find(start_marker)
end = code.find(end_marker)

if start == -1 or end == -1:
    raise ValueError("Could not find the Gemini retry section in rag_engine.py")

new_retry_section = '''# ============================================================
# GEMINI RETRY
# ============================================================

def generate_with_retry(prompt, max_retries=4):
    """
    Generate a Gemini response with intelligent retry handling.

    Temporary errors such as:
    - 429 RESOURCE_EXHAUSTED
    - 503 UNAVAILABLE

    are retried with exponential backoff.

    Permanent errors such as:
    - Invalid API key
    - Invalid request

    are raised immediately.
    """

    for attempt in range(max_retries):

        try:

            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt
            )

            return response.text.strip()

        except ServerError as e:

            # Server errors such as 503 can be temporary.
            if attempt < max_retries - 1:

                wait_time = 2 ** attempt

                print(
                    f"Gemini server temporarily unavailable. "
                    f"Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:
                raise e


        except ClientError as e:

            error_message = str(e)

            # Retry only quota/rate-limit errors.
            if (
                "429" in error_message
                or "RESOURCE_EXHAUSTED" in error_message
            ) and attempt < max_retries - 1:

                wait_time = 2 ** attempt

                print(
                    f"Gemini rate limit reached. "
                    f"Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:
                # Invalid API key, bad request, etc.
                # should not be retried.
                raise e


        except Exception as e:

            # Unknown errors are not blindly retried.
            raise e


    raise RuntimeError(
        "Gemini generation failed after all retry attempts."
    )


'''

# Replace only the retry section
updated_code = (
    code[:start]
    + new_retry_section
    + code[end:]
)

# Save updated engine
RAG_ENGINE_PATH.write_text(
    updated_code,
    encoding="utf-8"
)

print("=" * 80)
print("SMART GEMINI RETRY LOGIC UPDATED")
print("=" * 80)
print("✅ 503 ServerError → retries")
print("✅ 429 RESOURCE_EXHAUSTED → retries")
print("✅ Invalid API key → fails immediately")
print("✅ Invalid request → fails immediately")
print("✅ Unknown errors → fail immediately")
print("=" * 80)
print("Location:", RAG_ENGINE_PATH)

SMART GEMINI RETRY LOGIC UPDATED
✅ 503 ServerError → retries
✅ 429 RESOURCE_EXHAUSTED → retries
✅ Invalid API key → fails immediately
✅ Invalid request → fails immediately
✅ Unknown errors → fail immediately
Location: /content/drive/MyDrive/Enterprise_RAG_Assistant/src/rag_engine.py


In [25]:
# ============================================================
# STEP 10B-26 — FINAL RAG BACKEND CLEANUP
# ============================================================

from pathlib import Path

RAG_ENGINE_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/src/rag_engine.py"
)

# ------------------------------------------------------------
# Read the current RAG engine
# ------------------------------------------------------------

code = RAG_ENGINE_PATH.read_text(
    encoding="utf-8"
)

# ------------------------------------------------------------
# Update answer_question()
#
# We keep the existing retrieval and Gemini logic,
# but make the returned structure clean and predictable
# for the future frontend.
# ------------------------------------------------------------

start_marker = "# ============================================================\n# COMPLETE RAG PIPELINE\n# ============================================================"

start = code.find(start_marker)

if start == -1:
    raise ValueError(
        "Could not find COMPLETE RAG PIPELINE section."
    )

# Everything before the complete pipeline stays unchanged.
base_code = code[:start]

# ------------------------------------------------------------
# New complete RAG pipeline
# ------------------------------------------------------------

new_pipeline = r'''
# ============================================================
# COMPLETE RAG PIPELINE
# ============================================================

def answer_question(question, top_k=3):
    """
    Complete Enterprise RAG pipeline.

    Returns a consistent dictionary containing:

    - question
    - answer
    - sources
    - retrieved_count
    """

    # --------------------------------------------------------
    # Validate question
    # --------------------------------------------------------

    if not question or not question.strip():

        return {
            "question": question,
            "answer": "Please enter a question.",
            "sources": [],
            "retrieved_count": 0
        }

    # Remove unnecessary whitespace.
    question = question.strip()

    # --------------------------------------------------------
    # STEP 1 — RETRIEVE
    # --------------------------------------------------------

    retrieved_results = retrieve(
        question,
        top_k=top_k
    )

    # --------------------------------------------------------
    # STEP 2 — NO RELEVANT INFORMATION
    # --------------------------------------------------------

    if not retrieved_results:

        return {
            "question": question,

            "answer": (
                "This information is not available in "
                "the provided company documents."
            ),

            "sources": [],

            "retrieved_count": 0
        }

    # --------------------------------------------------------
    # STEP 3 — BUILD GROUNDED PROMPT
    # --------------------------------------------------------

    prompt = build_rag_prompt(
        question,
        retrieved_results
    )

    # --------------------------------------------------------
    # STEP 4 — GENERATE ANSWER
    # --------------------------------------------------------

    try:

        answer = generate_with_retry(
            prompt
        )

    except Exception as e:

        # Keep the error away from the user-facing answer.
        print(
            "Gemini generation error:",
            type(e).__name__,
            str(e)
        )

        return {
            "question": question,

            "answer": (
                "I’m temporarily unable to generate an answer. "
                "Please try again later."
            ),

            "sources": [],

            "retrieved_count": len(retrieved_results)
        }

    # --------------------------------------------------------
    # STEP 5 — PREPARE SOURCE INFORMATION
    # --------------------------------------------------------

    sources = []

    for result in retrieved_results:

        sources.append({
            "document": result["document"],
            "chunk_id": result["chunk_id"],
            "score": round(
                float(result["score"]),
                4
            )
        })

    # --------------------------------------------------------
    # STEP 6 — RETURN FRONTEND-FRIENDLY RESPONSE
    # --------------------------------------------------------

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_count": len(retrieved_results)
    }
'''

# ------------------------------------------------------------
# Save updated RAG engine
# ------------------------------------------------------------

RAG_ENGINE_PATH.write_text(
    base_code + new_pipeline,
    encoding="utf-8"
)

print("=" * 80)
print("FINAL RAG BACKEND CLEANUP COMPLETE")
print("=" * 80)

print("✅ Empty-question handling added")
print("✅ Clean response structure added")
print("✅ Source metadata preserved")
print("✅ Retrieved count added")
print("✅ Gemini failure handled safely")
print("✅ Frontend-friendly response created")

print("\nResponse structure:")
print("""
{
    "question": "...",
    "answer": "...",
    "sources": [...],
    "retrieved_count": 3
}
""")

print("Location:", RAG_ENGINE_PATH)

print("=" * 80)

FINAL RAG BACKEND CLEANUP COMPLETE
✅ Empty-question handling added
✅ Clean response structure added
✅ Source metadata preserved
✅ Retrieved count added
✅ Gemini failure handled safely
✅ Frontend-friendly response created

Response structure:

{
    "question": "...",
    "answer": "...",
    "sources": [...],
    "retrieved_count": 3
}

Location: /content/drive/MyDrive/Enterprise_RAG_Assistant/src/rag_engine.py


In [26]:
# ============================================================
# STEP 10B-28 — CREATE PROFESSIONAL STREAMLIT APPLICATION
# ============================================================

from pathlib import Path

APP_DIR = Path("/content/drive/MyDrive/Enterprise_RAG_Assistant/app")
APP_DIR.mkdir(parents=True, exist_ok=True)

APP_FILE = APP_DIR / "app.py"

app_code = r'''
import sys
from pathlib import Path

import streamlit as st

# ============================================================
# PATH CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant"
)

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


# ============================================================
# IMPORT RAG ENGINE
# ============================================================

from rag_engine import answer_question


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="TechNova AI Assistant",
    page_icon="🤖",
    layout="wide",
    initial_sidebar_state="expanded"
)


# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown(
    """
    <style>

    /* Main page */
    .main {
        padding-top: 1rem;
    }

    /* Header */
    .header-container {
        padding: 1rem 0 1.5rem 0;
    }

    .title {
        font-size: 2.4rem;
        font-weight: 700;
        margin-bottom: 0.2rem;
    }

    .subtitle {
        font-size: 1.05rem;
        color: #6b7280;
    }

    /* Answer box */
    .answer-box {
        padding: 1.5rem;
        border-radius: 12px;
        border: 1px solid #e5e7eb;
        background: #ffffff;
        margin-top: 1rem;
        margin-bottom: 1rem;
    }

    .answer-title {
        font-size: 1.1rem;
        font-weight: 650;
        margin-bottom: 0.8rem;
    }

    /* Source cards */
    .source-card {
        padding: 0.9rem 1rem;
        border-radius: 10px;
        border: 1px solid #e5e7eb;
        background: #f9fafb;
        margin-bottom: 0.6rem;
    }

    .source-name {
        font-weight: 600;
    }

    .source-score {
        color: #6b7280;
        font-size: 0.85rem;
    }

    /* Sidebar */
    .sidebar-title {
        font-size: 1.3rem;
        font-weight: 700;
        margin-bottom: 1rem;
    }

    .info-box {
        padding: 1rem;
        border-radius: 10px;
        border: 1px solid #e5e7eb;
        background: #f9fafb;
        margin-top: 1rem;
    }

    </style>
    """,
    unsafe_allow_html=True
)


# ============================================================
# SESSION STATE
# ============================================================

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    st.markdown(
        '<div class="sidebar-title">🏢 TechNova Solutions</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        """
        ### 🤖 Enterprise AI Assistant

        Ask questions about internal company policies and
        organizational documents.

        **Knowledge sources include:**
        - Leave Policy
        - Attendance Policy
        - Work From Home Policy
        - Employee Handbook
        - IT Security Policy
        - Acceptable Use Policy
        """,
    )

    st.divider()

    if st.button(
        "🗑️ Clear Conversation",
        use_container_width=True
    ):
        st.session_state.chat_history = []
        st.rerun()

    st.markdown(
        """
        <div class="info-box">
        <b>🔒 Enterprise Knowledge</b><br><br>
        Answers are generated using the organization's
        indexed internal documents.
        </div>
        """,
        unsafe_allow_html=True
    )


# ============================================================
# MAIN HEADER
# ============================================================

st.markdown(
    """
    <div class="header-container">
        <div class="title">🤖 TechNova AI Assistant</div>
        <div class="subtitle">
            Enterprise RAG Assistant for internal organizational knowledge
        </div>
    </div>
    """,
    unsafe_allow_html=True
)

st.divider()


# ============================================================
# WELCOME MESSAGE
# ============================================================

if not st.session_state.chat_history:

    st.info(
        """
        👋 **Welcome!**

        Ask me about TechNova Solutions policies, attendance,
        leave, work-from-home rules, security policies, and
        other internal organizational information.
        """
    )

    st.markdown("### 💡 Try asking")

    example_columns = st.columns(3)

    examples = [
        "How many annual leave days do employees get?",
        "What are the core working hours?",
        "How many consecutive days can employees work from home?"
    ]

    for col, example in zip(example_columns, examples):
        with col:
            if st.button(
                example,
                use_container_width=True
            ):
                st.session_state.selected_question = example


# ============================================================
# CHAT HISTORY
# ============================================================

for message in st.session_state.chat_history:

    with st.chat_message(message["role"]):

        st.markdown(message["content"])

        if message["role"] == "assistant":

            sources = message.get("sources", [])

            if sources:

                with st.expander(
                    f"📚 Sources ({len(sources)})"
                ):

                    for source in sources:

                        document = source.get(
                            "document",
                            "Unknown document"
                        )

                        score = source.get(
                            "score",
                            0
                        )

                        st.markdown(
                            f"""
                            <div class="source-card">
                                <div class="source-name">
                                    📄 {document}
                                </div>
                                <div class="source-score">
                                    Retrieval score: {score:.4f}
                                </div>
                            </div>
                            """,
                            unsafe_allow_html=True
                        )


# ============================================================
# QUESTION INPUT
# ============================================================

selected_question = st.session_state.pop(
    "selected_question",
    ""
)

question = st.chat_input(
    "Ask a question about company policies..."
)

if selected_question:
    question = selected_question


# ============================================================
# PROCESS QUESTION
# ============================================================

if question:

    # Display user question
    st.session_state.chat_history.append(
        {
            "role": "user",
            "content": question
        }
    )

    with st.chat_message("user"):
        st.markdown(question)

    # Generate answer
    with st.chat_message("assistant"):

        with st.spinner("🔎 Searching company knowledge..."):

            result = answer_question(question)

        answer = result.get(
            "answer",
            "No answer was generated."
        )

        sources = result.get(
            "sources",
            []
        )

        retrieved_count = result.get(
            "retrieved_count",
            0
        )

        st.markdown(
            '<div class="answer-box">',
            unsafe_allow_html=True
        )

        st.markdown(
            '<div class="answer-title">🤖 Answer</div>',
            unsafe_allow_html=True
        )

        st.markdown(answer)

        st.markdown(
            '</div>',
            unsafe_allow_html=True
        )

        # Source information
        if sources:

            with st.expander(
                f"📚 Sources ({len(sources)})"
            ):

                for source in sources:

                    document = source.get(
                        "document",
                        "Unknown document"
                    )

                    score = source.get(
                        "score",
                        0
                    )

                    st.markdown(
                        f"""
                        <div class="source-card">
                            <div class="source-name">
                                📄 {document}
                            </div>
                            <div class="source-score">
                                Retrieval score: {score:.4f}
                            </div>
                        </div>
                        """,
                        unsafe_allow_html=True
                    )

        # Retrieval information
        if retrieved_count > 0:

            st.caption(
                f"🔍 Retrieved {retrieved_count} relevant "
                f"document chunk(s)"
            )

        # Save assistant response
        st.session_state.chat_history.append(
            {
                "role": "assistant",
                "content": answer,
                "sources": sources,
                "retrieved_count": retrieved_count
            }
        )
'''

APP_FILE.write_text(app_code, encoding="utf-8")

print("=" * 80)
print("STEP 10B-28 — STREAMLIT APPLICATION CREATED")
print("=" * 80)

print(f"Application file:")
print(APP_FILE)

print()
print("✅ Professional UI created")
print("✅ Chat interface added")
print("✅ Conversation history added")
print("✅ Source display added")
print("✅ Retrieval information added")
print("✅ Clear conversation button added")
print("✅ Example questions added")
print("✅ Existing RAG engine preserved")

print("=" * 80)
print("NEXT: TEST THE APPLICATION LOCALLY")
print("=" * 80)

STEP 10B-28 — STREAMLIT APPLICATION CREATED
Application file:
/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py

✅ Professional UI created
✅ Chat interface added
✅ Conversation history added
✅ Source display added
✅ Retrieval information added
✅ Clear conversation button added
✅ Example questions added
✅ Existing RAG engine preserved
NEXT: TEST THE APPLICATION LOCALLY


In [27]:
# ============================================================
# STEP 10B-31 — SWITCH RAG ENGINE TO GEMINI 3.5 FLASH-LITE
# ============================================================

from pathlib import Path

# Path to the RAG engine
rag_file = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/src/rag_engine.py"
)

# Read the existing file
content = rag_file.read_text()

# Replace the old Gemini model with the working model
old_model = "gemini-3.6-flash"
new_model = "gemini-3.5-flash-lite"

if old_model in content:

    content = content.replace(old_model, new_model)

    rag_file.write_text(content)

    print("=" * 80)
    print("GEMINI MODEL UPDATED")
    print("=" * 80)
    print()
    print("Old model:", old_model)
    print("New model:", new_model)
    print()
    print("✅ rag_engine.py updated successfully.")
    print("=" * 80)

else:

    print("=" * 80)
    print("MODEL CHECK")
    print("=" * 80)

    if new_model in content:
        print("✅ Gemini 3.5 Flash-Lite is already configured.")
    else:
        print("⚠️ Expected Gemini model string was not found.")

    print("=" * 80)

GEMINI MODEL UPDATED

Old model: gemini-3.6-flash
New model: gemini-3.5-flash-lite

✅ rag_engine.py updated successfully.


In [28]:
# ============================================================
# STEP 10B-35 — UPGRADE ENTERPRISE RAG UI
# ============================================================

from pathlib import Path

APP_FILE = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py"
)

app_code = r'''
import sys
from pathlib import Path

import streamlit as st


# ============================================================
# PATH CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


# ============================================================
# RAG ENGINE
# ============================================================

from rag_engine import answer_question


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="TechNova AI",
    page_icon="🤖",
    layout="wide",
    initial_sidebar_state="expanded",
)


# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown(
    """
    <style>

    /* -------------------------------------------------------
       GLOBAL
    ------------------------------------------------------- */

    .stApp {
        background:
            radial-gradient(
                circle at 20% 10%,
                rgba(99, 102, 241, 0.12),
                transparent 30%
            ),
            radial-gradient(
                circle at 85% 15%,
                rgba(14, 165, 233, 0.10),
                transparent 30%
            ),
            #0b0d12;
    }

    .main .block-container {
        max-width: 1200px;
        padding-top: 2rem;
        padding-bottom: 5rem;
    }


    /* -------------------------------------------------------
       SIDEBAR
       ------------------------------------------------------- */

    section[data-testid="stSidebar"] {
        background:
            linear-gradient(
                180deg,
                #151821 0%,
                #101219 100%
            );
        border-right: 1px solid rgba(255,255,255,0.07);
    }

    section[data-testid="stSidebar"] .block-container {
        padding: 2rem 1.25rem;
    }

    .brand {
        display: flex;
        align-items: center;
        gap: 12px;
        margin-bottom: 1.8rem;
    }

    .brand-icon {
        width: 45px;
        height: 45px;
        border-radius: 13px;

        display: flex;
        align-items: center;
        justify-content: center;

        font-size: 23px;

        background:
            linear-gradient(
                135deg,
                #6366f1,
                #06b6d4
            );

        box-shadow:
            0 10px 30px rgba(99,102,241,0.25);
    }

    .brand-title {
        font-size: 20px;
        font-weight: 750;
        color: #f8fafc;
        line-height: 1.1;
    }

    .brand-subtitle {
        font-size: 11px;
        color: #94a3b8;
        margin-top: 4px;
    }

    .sidebar-heading {
        font-size: 12px;
        font-weight: 700;
        text-transform: uppercase;
        letter-spacing: 0.08em;
        color: #64748b;
        margin-top: 1.8rem;
        margin-bottom: 0.8rem;
    }

    .source-item {
        display: flex;
        align-items: center;
        gap: 10px;

        padding: 10px 11px;
        margin: 6px 0;

        border-radius: 10px;

        background: rgba(255,255,255,0.025);
        border: 1px solid rgba(255,255,255,0.045);

        color: #cbd5e1;
        font-size: 13px;
    }

    .source-dot {
        width: 7px;
        height: 7px;
        border-radius: 50%;
        background: #22c55e;
        box-shadow: 0 0 10px rgba(34,197,94,0.6);
    }

    .sidebar-footer {
        margin-top: 2rem;
        padding-top: 1rem;

        border-top: 1px solid rgba(255,255,255,0.07);

        font-size: 11px;
        color: #64748b;
        line-height: 1.6;
    }


    /* -------------------------------------------------------
       HERO
       ------------------------------------------------------- */

    .hero {
        padding: 2.5rem 2.5rem 2.2rem 2.5rem;

        border-radius: 24px;

        background:
            linear-gradient(
                135deg,
                rgba(30,41,59,0.92),
                rgba(15,23,42,0.72)
            );

        border: 1px solid rgba(148,163,184,0.12);

        box-shadow:
            0 25px 80px rgba(0,0,0,0.28);

        margin-bottom: 1.6rem;
    }

    .hero-badge {
        display: inline-flex;
        align-items: center;
        gap: 7px;

        padding: 6px 11px;

        border-radius: 999px;

        background: rgba(34,197,94,0.09);
        border: 1px solid rgba(34,197,94,0.18);

        color: #86efac;
        font-size: 11px;
        font-weight: 700;

        text-transform: uppercase;
        letter-spacing: 0.07em;
    }

    .hero-title {
        font-size: clamp(34px, 5vw, 54px);
        font-weight: 800;

        line-height: 1.05;

        margin: 18px 0 12px 0;

        background:
            linear-gradient(
                90deg,
                #ffffff,
                #c7d2fe,
                #67e8f9
            );

        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
    }

    .hero-text {
        color: #94a3b8;
        font-size: 16px;
        line-height: 1.7;
        max-width: 750px;
    }


    /* -------------------------------------------------------
       STATUS CARDS
       ------------------------------------------------------- */

    .status-card {
        padding: 16px;

        border-radius: 16px;

        background: rgba(15,23,42,0.68);
        border: 1px solid rgba(148,163,184,0.10);

        margin-bottom: 1rem;
    }

    .status-label {
        color: #64748b;
        font-size: 11px;
        text-transform: uppercase;
        letter-spacing: 0.07em;
        font-weight: 700;
    }

    .status-value {
        color: #e2e8f0;
        font-size: 14px;
        font-weight: 650;
        margin-top: 6px;
    }


    /* -------------------------------------------------------
       SECTION TITLES
       ------------------------------------------------------- */

    .section-title {
        color: #f1f5f9;
        font-size: 17px;
        font-weight: 750;
        margin-top: 1.5rem;
        margin-bottom: 0.7rem;
    }


    /* -------------------------------------------------------
       CHAT
       ------------------------------------------------------- */

    .user-message {
        background:
            linear-gradient(
                135deg,
                rgba(79,70,229,0.18),
                rgba(59,130,246,0.08)
            );

        border: 1px solid rgba(99,102,241,0.18);

        border-radius: 18px 18px 5px 18px;

        padding: 15px 18px;

        margin: 15px 0 10px auto;

        max-width: 88%;

        color: #e2e8f0;
        line-height: 1.6;
    }

    .assistant-message {
        background:
            linear-gradient(
                135deg,
                rgba(30,41,59,0.85),
                rgba(15,23,42,0.78)
            );

        border: 1px solid rgba(148,163,184,0.10);

        border-radius: 5px 18px 18px 18px;

        padding: 18px 20px;

        margin: 8px auto 15px 0;

        max-width: 92%;

        color: #e2e8f0;
        line-height: 1.7;

        box-shadow:
            0 12px 35px rgba(0,0,0,0.15);
    }

    .message-label {
        font-size: 10px;
        font-weight: 800;
        text-transform: uppercase;
        letter-spacing: 0.08em;

        color: #64748b;

        margin-bottom: 8px;
    }


    /* -------------------------------------------------------
       SOURCE CARDS
       ------------------------------------------------------- */

    .source-card {
        padding: 13px 15px;

        margin: 7px 0;

        border-radius: 12px;

        background: rgba(2,6,23,0.48);

        border: 1px solid rgba(148,163,184,0.08);
    }

    .source-name {
        color: #c7d2fe;
        font-weight: 650;
        font-size: 13px;
    }

    .source-score {
        color: #64748b;
        font-size: 11px;
        margin-top: 4px;
    }


    /* -------------------------------------------------------
       WELCOME CARD
       ------------------------------------------------------- */

    .welcome {
        text-align: center;

        padding: 3rem 1rem 2rem 1rem;

        color: #94a3b8;
    }

    .welcome-icon {
        font-size: 46px;
        margin-bottom: 12px;
    }

    .welcome-title {
        color: #e2e8f0;
        font-size: 22px;
        font-weight: 750;
    }

    .welcome-text {
        max-width: 620px;
        margin: 8px auto;

        color: #64748b;
        line-height: 1.6;
    }


    /* -------------------------------------------------------
       INPUT
       ------------------------------------------------------- */

    div[data-testid="stChatInput"] {
        border-radius: 18px;
    }

    div[data-testid="stChatInput"] textarea {
        border-radius: 18px !important;
    }


    /* -------------------------------------------------------
       BUTTONS
       ------------------------------------------------------- */

    .stButton > button {
        border-radius: 11px;

        border: 1px solid rgba(148,163,184,0.12);

        background: rgba(30,41,59,0.55);

        color: #cbd5e1;

        transition: all 0.2s ease;
    }

    .stButton > button:hover {
        border-color: rgba(129,140,248,0.45);

        background: rgba(79,70,229,0.13);

        color: #ffffff;

        transform: translateY(-1px);
    }

    </style>
    """,
    unsafe_allow_html=True,
)


# ============================================================
# SESSION STATE
# ============================================================

if "messages" not in st.session_state:
    st.session_state.messages = []


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    st.markdown(
        """
        <div class="brand">
            <div class="brand-icon">🤖</div>
            <div>
                <div class="brand-title">TechNova AI</div>
                <div class="brand-subtitle">Enterprise Knowledge Assistant</div>
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    st.markdown(
        """
        <div class="status-card">
            <div class="status-label">System Status</div>
            <div class="status-value">🟢 AI system online</div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    st.markdown(
        '<div class="sidebar-heading">Knowledge Base</div>',
        unsafe_allow_html=True,
    )

    documents = [
        "Leave Policy",
        "Attendance Policy",
        "Work From Home Policy",
        "Employee Handbook",
        "IT Security Policy",
        "Acceptable Use Policy",
    ]

    for document in documents:
        st.markdown(
            f"""
            <div class="source-item">
                <span class="source-dot"></span>
                <span>{document}</span>
            </div>
            """,
            unsafe_allow_html=True,
        )

    st.markdown(
        '<div class="sidebar-heading">Assistant Capabilities</div>',
        unsafe_allow_html=True,
    )

    st.markdown(
        """
        <div class="source-item">🔎 Semantic document search</div>
        <div class="source-item">📚 Source attribution</div>
        <div class="source-item">🛡️ Grounded responses</div>
        <div class="source-item">💬 Conversation history</div>
        """,
        unsafe_allow_html=True,
    )

    st.markdown("---")

    if st.button(
        "🗑️  Clear Conversation",
        use_container_width=True,
    ):
        st.session_state.messages = []
        st.rerun()

    st.markdown(
        """
        <div class="sidebar-footer">
            TechNova Solutions<br>
            Enterprise Document Intelligence<br><br>
            Powered by Retrieval-Augmented Generation
        </div>
        """,
        unsafe_allow_html=True,
    )


# ============================================================
# MAIN HERO
# ============================================================

st.markdown(
    """
    <div class="hero">

        <div class="hero-badge">
            ● Secure Knowledge Assistant
        </div>

        <div class="hero-title">
            Ask your company knowledge.
        </div>

        <div class="hero-text">
            Search internal policies, employee documentation,
            and organizational knowledge using natural language.
            Answers are grounded in the retrieved company documents.
        </div>

    </div>
    """,
    unsafe_allow_html=True,
)


# ============================================================
# EXAMPLE QUESTIONS
# ============================================================

st.markdown(
    '<div class="section-title">✨ Try asking</div>',
    unsafe_allow_html=True,
)

example_questions = [
    "How many annual leave days do employees get?",
    "What are the core working hours?",
    "How many consecutive days can employees work from home?",
]

cols = st.columns(3)

for i, question in enumerate(example_questions):

    with cols[i]:

        if st.button(
            question,
            key=f"example_{i}",
            use_container_width=True,
        ):
            st.session_state.pending_question = question
            st.rerun()


# ============================================================
# PENDING EXAMPLE QUESTION
# ============================================================

pending_question = st.session_state.pop(
    "pending_question",
    None,
)


# ============================================================
# CHAT HISTORY
# ============================================================

if not st.session_state.messages:

    st.markdown(
        """
        <div class="welcome">

            <div class="welcome-icon">🧠</div>

            <div class="welcome-title">
                Your enterprise knowledge, one question away.
            </div>

            <div class="welcome-text">
                Ask about company policies, attendance,
                leave, work-from-home rules, security,
                or information contained in the knowledge base.
            </div>

        </div>
        """,
        unsafe_allow_html=True,
    )


for message in st.session_state.messages:

    role = message["role"]
    content = message["content"]

    if role == "user":

        st.markdown(
            f"""
            <div class="user-message">
                <div class="message-label">You</div>
                {content}
            </div>
            """,
            unsafe_allow_html=True,
        )

    else:

        st.markdown(
            f"""
            <div class="assistant-message">
                <div class="message-label">🤖 TechNova AI</div>
                {content}
            </div>
            """,
            unsafe_allow_html=True,
        )

        sources = message.get("sources", [])

        if sources:

            with st.expander(
                f"📚 Sources ({len(sources)})",
                expanded=False,
            ):

                for source in sources:

                    if isinstance(source, dict):

                        document = source.get(
                            "document",
                            source.get(
                                "source",
                                "Unknown document",
                            ),
                        )

                        score = source.get(
                            "score",
                            None,
                        )

                        score_text = ""

                        if isinstance(score, (float, int)):
                            score_text = (
                                f"Similarity score: {score:.4f}"
                            )

                        st.markdown(
                            f"""
                            <div class="source-card">

                                <div class="source-name">
                                    📄 {document}
                                </div>

                                <div class="source-score">
                                    {score_text}
                                </div>

                            </div>
                            """,
                            unsafe_allow_html=True,
                        )

                    else:

                        st.markdown(
                            f"📄 {source}"
                        )

        retrieved_count = message.get(
            "retrieved_count",
            0,
        )

        if retrieved_count:

            st.caption(
                f"🔎 Retrieved {retrieved_count} "
                f"relevant document chunk(s)"
            )


# ============================================================
# QUESTION INPUT
# ============================================================

question = st.chat_input(
    "Ask anything about company policies..."
)


# Use example question if selected
if pending_question:
    question = pending_question


# ============================================================
# PROCESS QUESTION
# ============================================================

if question:

    # Add user message
    st.session_state.messages.append(
        {
            "role": "user",
            "content": question,
        }
    )

    # Generate answer
    with st.spinner("Searching company knowledge..."):

        result = answer_question(question)

    answer = result.get(
        "answer",
        "I couldn't generate an answer.",
    )

    sources = result.get(
        "sources",
        [],
    )

    retrieved_count = result.get(
        "retrieved_count",
        0,
    )

    # Add assistant response
    st.session_state.messages.append(
        {
            "role": "assistant",
            "content": answer,
            "sources": sources,
            "retrieved_count": retrieved_count,
        }
    )

    # Refresh UI
    st.rerun()


# ============================================================
# FOOTER
# ============================================================

st.markdown(
    """
    <div style="
        text-align:center;
        margin-top:3rem;
        padding-top:1.2rem;
        border-top:1px solid rgba(255,255,255,0.06);
        color:#475569;
        font-size:11px;
    ">
        TechNova AI • Enterprise Document Intelligence
        • Retrieval-Augmented Generation
    </div>
    """,
    unsafe_allow_html=True,
)
'''

# Write the new application
APP_FILE.write_text(app_code)

print("=" * 80)
print("STEP 10B-35 — UI UPGRADE COMPLETE")
print("=" * 80)
print()
print("Application:", APP_FILE)
print()
print("✅ Modern enterprise theme added")
print("✅ TechNova AI branding added")
print("✅ Hero section added")
print("✅ Knowledge-base sidebar redesigned")
print("✅ Capability indicators added")
print("✅ Chat interface redesigned")
print("✅ Source cards redesigned")
print("✅ Example question buttons added")
print("✅ Conversation history preserved")
print("✅ Clear conversation preserved")
print("✅ RAG backend unchanged")
print()
print("=" * 80)
print("NEXT: RESTART STREAMLIT")
print("=" * 80)

STEP 10B-35 — UI UPGRADE COMPLETE

Application: /content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py

✅ Modern enterprise theme added
✅ TechNova AI branding added
✅ Hero section added
✅ Knowledge-base sidebar redesigned
✅ Capability indicators added
✅ Chat interface redesigned
✅ Source cards redesigned
✅ Example question buttons added
✅ Conversation history preserved
✅ Clear conversation preserved
✅ RAG backend unchanged

NEXT: RESTART STREAMLIT


In [29]:
from pathlib import Path

# ========================================================
# STEP 10B-58
# ADD OPTIONAL DOCUMENT FILTERING TO RETRIEVAL
# ========================================================

RAG_FILE = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/src/rag_engine.py"
)

content = RAG_FILE.read_text(encoding="utf-8")

# --------------------------------------------------------
# Find the current retrieve() function
# --------------------------------------------------------

start = content.find("def retrieve(")

if start == -1:
    print("❌ retrieve() function was not found.")
else:

    # Find the next major section after retrieve()
    end = content.find(
        "# ============================================================\n# RAG PROMPT",
        start
    )

    if end == -1:
        print("❌ Could not locate the end of retrieve().")
    else:

        old_function = content[start:end]

        new_function = '''def retrieve(
    query,
    top_k=3,
    document_name=None
):
    """
    Retrieve the most relevant document chunks for a query.

    Parameters:
    - query: User's question.
    - top_k: Maximum number of chunks to retrieve.
    - document_name: Optional document filename.

    If document_name is provided, only chunks belonging
    to that document are considered.

    If document_name is None, the entire knowledge base
    is searched normally.
    """

    # Convert the user's question into an embedding.
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # ----------------------------------------------------
    # SEARCH MORE RESULTS WHEN DOCUMENT FILTERING
    # ----------------------------------------------------
    #
    # We search the complete index first and then filter
    # the results by document name.
    #
    # This avoids rebuilding the FAISS index.
    # ----------------------------------------------------

    search_k = index.ntotal

    scores, indices = index.search(
        query_embedding,
        search_k
    )

    results = []

    # Process retrieved results.
    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        # FAISS can return -1 for invalid results.
        if idx < 0:
            continue

        score = float(score)

        # Ignore low-relevance chunks.
        if score < RELEVANCE_THRESHOLD:
            continue

        # Get corresponding chunk.
        chunk = chunks[idx]

        # ------------------------------------------------
        # DOCUMENT FILTER
        # ------------------------------------------------
        #
        # If a document name was supplied, ignore chunks
        # belonging to other documents.
        # ------------------------------------------------

        if (
            document_name is not None
            and chunk["document"] != document_name
        ):
            continue

        results.append({
            "document": chunk["document"],
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "score": score
        })

        # Stop once we have enough relevant results.
        if len(results) >= top_k:
            break

    return results


'''

        # Replace only retrieve()
        content = (
            content[:start]
            + new_function
            + content[end:]
        )

        RAG_FILE.write_text(
            content,
            encoding="utf-8"
        )

        print("=" * 60)
        print("STEP 10B-58 COMPLETE")
        print("=" * 60)
        print("✅ retrieve() updated")
        print("✅ Optional document_name parameter added")
        print("✅ Document filtering added")
        print("✅ Existing FAISS index preserved")
        print("✅ General knowledge-base search preserved")
        print("✅ Relevance threshold preserved")
        print("✅ app.py not modified")
        print("============================================================")

STEP 10B-58 COMPLETE
✅ retrieve() updated
✅ Optional document_name parameter added
✅ Document filtering added
✅ Existing FAISS index preserved
✅ General knowledge-base search preserved
✅ Relevance threshold preserved
✅ app.py not modified


In [31]:
from pathlib import Path

RAG_ENGINE_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/src/rag_engine.py"
)

print("File exists:", RAG_ENGINE_PATH.exists())
print("File:", RAG_ENGINE_PATH)

File exists: True
File: /content/drive/MyDrive/Enterprise_RAG_Assistant/src/rag_engine.py


In [32]:
from pathlib import Path

RAG_ENGINE_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/src/rag_engine.py"
)

print("=" * 60)
print("CURRENT rag_engine.py")
print("=" * 60)

with open(RAG_ENGINE_PATH, "r", encoding="utf-8") as f:
    rag_code = f.read()

print(rag_code)

CURRENT rag_engine.py

# ============================================================
# TECHNOVA ENTERPRISE RAG ENGINE
# ============================================================

from pathlib import Path
import pickle
import time

import faiss
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai.errors import ClientError, ServerError


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(__file__).resolve().parent.parent

VECTORSTORE_DIR = PROJECT_DIR / "vectorstore"

FAISS_PATH = VECTORSTORE_DIR / "tech_nova.index"
CHUNKS_PATH = VECTORSTORE_DIR / "chunks.pkl"


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "all-MiniLM-L6-v2"

GEMINI_MODEL = "gemini-3.5-flash-lite"

# Minimum FAISS similarity score required for a result
# to be con

In [35]:
from pathlib import Path

RAG_ENGINE_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/src/rag_engine.py"
)

code = RAG_ENGINE_PATH.read_text(encoding="utf-8")

# 1. Replace Gemini client initialization
code = code.replace(
    """# ============================================================
# GEMINI CLIENT
# ============================================================

client = genai.Client()
""",
    """# ============================================================
# GEMINI CLIENT
# ============================================================

def get_gemini_client():
    \"\"\"Create Gemini client only when generation is requested.\"\"\"
    import os

    api_key = os.environ.get("GEMINI_API_KEY")

    if not api_key:
        raise ValueError(
            "GEMINI_API_KEY is not configured. "
            "Please configure your Gemini API key before generating answers."
        )

    return genai.Client(api_key=api_key)
"""
)

# 2. Replace Gemini generation call
code = code.replace(
    """            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt
            )
""",
    """            client = get_gemini_client()

            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt
            )
"""
)

# 3. Add document_name support to answer_question
code = code.replace(
    "def answer_question(question, top_k=3):",
    "def answer_question(question, top_k=3, document_name=None):"
)

# 4. Pass document_name to retrieve()
code = code.replace(
    """    retrieved_results = retrieve(
        question,
        top_k=top_k
    )
""",
    """    retrieved_results = retrieve(
        question,
        top_k=top_k,
        document_name=document_name
    )
"""
)

# Save directly to Google Drive
RAG_ENGINE_PATH.write_text(code, encoding="utf-8")

print("=" * 60)
print("✅ rag_engine.py UPDATED")
print("=" * 60)
print("✅ Gemini client changed to lazy loading")
print("✅ API key is only needed when generating")
print("✅ answer_question() supports document filtering")
print("✅ Changes saved directly to Google Drive")
print()
print(RAG_ENGINE_PATH)

✅ rag_engine.py UPDATED
✅ Gemini client changed to lazy loading
✅ API key is only needed when generating
✅ answer_question() supports document filtering
✅ Changes saved directly to Google Drive

/content/drive/MyDrive/Enterprise_RAG_Assistant/src/rag_engine.py


In [36]:
import sys
import inspect

# Remove old imported version if it exists
if "src.rag_engine" in sys.modules:
    del sys.modules["src.rag_engine"]

import src.rag_engine as rag_engine

print()
print("=" * 60)
print("✅ RAG ENGINE RELOADED SUCCESSFULLY")
print("=" * 60)

print()
print("retrieve() signature:")
print(inspect.signature(rag_engine.retrieve))

print()
print("answer_question() signature:")
print(inspect.signature(rag_engine.answer_question))

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded: all-MiniLM-L6-v2
Loading FAISS index...
FAISS index loaded!
Vectors: 20
Dimension: 384
Loading chunks...
Chunks loaded: 20

✅ RAG ENGINE RELOADED SUCCESSFULLY

retrieve() signature:
(query, top_k=3, document_name=None)

answer_question() signature:
(question, top_k=3, document_name=None)


In [37]:
# ========================================================
# STEP 10B-59A
# RELOAD UPDATED RAG ENGINE
# ========================================================

import importlib
import src.rag_engine as rag_engine

# Reload the Python module so Colab picks up
# the newly modified retrieve() function.
importlib.reload(rag_engine)

# Use the newly loaded function.
retrieve = rag_engine.retrieve

print("=" * 60)
print("RAG ENGINE RELOADED")
print("=" * 60)

print("✅ Updated retrieve() loaded successfully.")
print("✅ document_name parameter is now available.")

# Check the function signature
import inspect

print("\nCurrent retrieve() signature:")
print(inspect.signature(retrieve))

print("=" * 60)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded: all-MiniLM-L6-v2
Loading FAISS index...
FAISS index loaded!
Vectors: 20
Dimension: 384
Loading chunks...
Chunks loaded: 20
RAG ENGINE RELOADED
✅ Updated retrieve() loaded successfully.
✅ document_name parameter is now available.

Current retrieve() signature:
(query, top_k=3, document_name=None)


In [38]:
# ========================================================
# STEP 10B-59
# TEST DOCUMENT-SPECIFIC RETRIEVAL
# ========================================================

from src.rag_engine import retrieve

print("=" * 70)
print("DOCUMENT-SPECIFIC RETRIEVAL TEST")
print("=" * 70)

# --------------------------------------------------------
# TEST 1 — Leave Policy
# --------------------------------------------------------

question = "How many annual leave days are provided?"

results = retrieve(
    question,
    top_k=3,
    document_name="Leave_Policy.docx"
)

print("\nTEST 1 — LEAVE POLICY")
print("-" * 70)
print("Question:", question)
print("Requested document: Leave_Policy.docx")
print("Results:", len(results))

for i, result in enumerate(results, start=1):

    print(
        f"{i}. {result['document']} "
        f"| Score: {result['score']:.4f}"
    )

# Verify every result belongs to Leave Policy
if results and all(
    result["document"] == "Leave_Policy.docx"
    for result in results
):
    print("✅ All results belong to Leave_Policy.docx")
else:
    print("❌ Document filtering failed")


# --------------------------------------------------------
# TEST 2 — General Search
# --------------------------------------------------------

question = "How many annual leave days are provided?"

results_general = retrieve(
    question,
    top_k=3
)

print("\nTEST 2 — GENERAL KNOWLEDGE BASE")
print("-" * 70)
print("Question:", question)
print("Document filter: None")
print("Results:", len(results_general))

for i, result in enumerate(results_general, start=1):

    print(
        f"{i}. {result['document']} "
        f"| Score: {result['score']:.4f}"
    )

if results_general:
    print("✅ General retrieval still works")
else:
    print("❌ General retrieval failed")


# --------------------------------------------------------
# FINAL RESULT
# --------------------------------------------------------

print("\n" + "=" * 70)

if (
    results
    and all(
        result["document"] == "Leave_Policy.docx"
        for result in results
    )
    and results_general
):
    print("✅ DOCUMENT-SPECIFIC RETRIEVAL TEST PASSED")
else:
    print("❌ DOCUMENT-SPECIFIC RETRIEVAL TEST FAILED")

print("=" * 70)

DOCUMENT-SPECIFIC RETRIEVAL TEST

TEST 1 — LEAVE POLICY
----------------------------------------------------------------------
Question: How many annual leave days are provided?
Requested document: Leave_Policy.docx
Results: 3
1. Leave_Policy.docx | Score: 0.6500
2. Leave_Policy.docx | Score: 0.4426
3. Leave_Policy.docx | Score: 0.3491
✅ All results belong to Leave_Policy.docx

TEST 2 — GENERAL KNOWLEDGE BASE
----------------------------------------------------------------------
Question: How many annual leave days are provided?
Document filter: None
Results: 3
1. Leave_Policy.docx | Score: 0.6500
2. Leave_Policy.docx | Score: 0.4426
3. Work_From_Home_Policy.docx | Score: 0.3800
✅ General retrieval still works

✅ DOCUMENT-SPECIFIC RETRIEVAL TEST PASSED


In [39]:
# ============================================================
# STEP 10B-60 — RETRIEVAL VALIDATION
# ============================================================

from src.rag_engine import retrieve

print("=" * 60)
print("TEST 1 — GENERAL KNOWLEDGE BASE")
print("=" * 60)

question = "How many annual leave days are provided?"

results = retrieve(question, top_k=3)

print(f"Question: {question}")
print(f"Results: {len(results)}")
print()

for i, result in enumerate(results, start=1):
    print(
        f"{i}. {result['document']} | "
        f"Score: {result['score']:.4f}"
    )

print()
print("=" * 60)
print("TEST 2 — DOCUMENT-SPECIFIC RETRIEVAL")
print("=" * 60)

document_name = "Leave_Policy.docx"

results = retrieve(
    question,
    top_k=3,
    document_name=document_name
)

print(f"Question: {question}")
print(f"Requested document: {document_name}")
print(f"Results: {len(results)}")
print()

for i, result in enumerate(results, start=1):
    print(
        f"{i}. {result['document']} | "
        f"Score: {result['score']:.4f}"
    )

# Verify every result belongs to the requested document
if all(r["document"] == document_name for r in results):
    print()
    print("✅ DOCUMENT FILTER TEST PASSED")
else:
    print()
    print("❌ DOCUMENT FILTER TEST FAILED")

TEST 1 — GENERAL KNOWLEDGE BASE
Question: How many annual leave days are provided?
Results: 3

1. Leave_Policy.docx | Score: 0.6500
2. Leave_Policy.docx | Score: 0.4426
3. Work_From_Home_Policy.docx | Score: 0.3800

TEST 2 — DOCUMENT-SPECIFIC RETRIEVAL
Question: How many annual leave days are provided?
Requested document: Leave_Policy.docx
Results: 3

1. Leave_Policy.docx | Score: 0.6500
2. Leave_Policy.docx | Score: 0.4426
3. Leave_Policy.docx | Score: 0.3491

✅ DOCUMENT FILTER TEST PASSED


In [40]:
from google.colab import userdata
import os

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("✅ Gemini API key loaded from Colab Secrets")

✅ Gemini API key loaded from Colab Secrets


In [41]:
from src.rag_engine import answer_question

question = "How many annual leave days are provided?"

print("=" * 60)
print("TEST 3 — FULL RAG GENERATION")
print("=" * 60)
print()
print("Question:", question)
print()

result = answer_question(question)

print("Answer:")
print(result["answer"])

print()
print("Sources:")
for source in result["sources"]:
    print(
        f"- {source['document']} | "
        f"Chunk: {source['chunk_id']} | "
        f"Score: {source['score']}"
    )

print()
print("Retrieved count:", result["retrieved_count"])

TEST 3 — FULL RAG GENERATION

Question: How many annual leave days are provided?



Answer:
Based on `Leave_Policy.docx`, full-time employees are entitled to 20 days of annual leave per calendar year.

Sources:
- Leave_Policy.docx | Chunk: Leave_Policy.docx_chunk_0010 | Score: 0.65
- Leave_Policy.docx | Chunk: Leave_Policy.docx_chunk_0011 | Score: 0.4426
- Work_From_Home_Policy.docx | Chunk: Work_From_Home_Policy.docx_chunk_0018 | Score: 0.38

Retrieved count: 3


In [42]:
from src.rag_engine import answer_question

question = "How many annual leave days are provided?"
document = "Leave_Policy.docx"

print("=" * 60)
print("TEST 4 — DOCUMENT-SPECIFIC FULL RAG")
print("=" * 60)
print()
print("Question:", question)
print("Document:", document)
print()

result = answer_question(
    question,
    document_name=document
)

print("Answer:")
print(result["answer"])

print()
print("Sources:")

for source in result["sources"]:
    print(
        f"- {source['document']} | "
        f"Chunk: {source['chunk_id']} | "
        f"Score: {source['score']}"
    )

print()
print("Retrieved count:", result["retrieved_count"])

# Verify sources
if all(
    source["document"] == document
    for source in result["sources"]
):
    print()
    print("✅ DOCUMENT-SPECIFIC FULL RAG TEST PASSED")
else:
    print()
    print("❌ DOCUMENT-SPECIFIC FULL RAG TEST FAILED")

TEST 4 — DOCUMENT-SPECIFIC FULL RAG

Question: How many annual leave days are provided?
Document: Leave_Policy.docx

Answer:
Based on the provided documents (`Leave_Policy.docx`), full-time employees are entitled to 20 days of annual leave per calendar year.

Sources:
- Leave_Policy.docx | Chunk: Leave_Policy.docx_chunk_0010 | Score: 0.65
- Leave_Policy.docx | Chunk: Leave_Policy.docx_chunk_0011 | Score: 0.4426
- Leave_Policy.docx | Chunk: Leave_Policy.docx_chunk_0012 | Score: 0.3491

Retrieved count: 3

✅ DOCUMENT-SPECIFIC FULL RAG TEST PASSED


In [43]:
from pathlib import Path

APP_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py"
)

print("=" * 60)
print("APP FILE CHECK")
print("=" * 60)
print()
print("File exists:", APP_PATH.exists())
print("File:", APP_PATH)

if APP_PATH.exists():
    print("File size:", APP_PATH.stat().st_size, "bytes")

APP FILE CHECK

File exists: True
File: /content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py
File size: 18702 bytes


In [45]:
from pathlib import Path

APP_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py"
)

print("=" * 60)
print("CURRENT app.py")
print("=" * 60)

app_code = APP_PATH.read_text(encoding="utf-8")

print(app_code)

CURRENT app.py

import sys
from pathlib import Path

import streamlit as st


# ============================================================
# PATH CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


# ============================================================
# RAG ENGINE
# ============================================================

from rag_engine import answer_question


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="TechNova AI",
    page_icon="🤖",
    layout="wide",
    initial_sidebar_state="expanded",
)


# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown(
    """

In [46]:
from pathlib import Path

APP_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py"
)

code = APP_PATH.read_text(encoding="utf-8")

old_block = '''    documents = [
        "Leave Policy",
        "Attendance Policy",
        "Work From Home Policy",
        "Employee Handbook",
        "IT Security Policy",
        "Acceptable Use Policy",
    ]

    for document in documents:
        st.markdown(
            f"""
            <div class="source-item">
                <span class="source-dot"></span>
                <span>{document}</span>
            </div>
            """,
            unsafe_allow_html=True,
        )
'''

new_block = '''    documents = {
        "Leave Policy": "Leave_Policy.docx",
        "Attendance Policy": "Attendance_Policy.docx",
        "Work From Home Policy": "Work_From_Home_Policy.docx",
        "Employee Handbook": "Employee_Handbook.docx",
        "IT Security Policy": "IT_Security_Policy.docx",
        "Acceptable Use Policy": "Acceptable_Use_Policy.docx",
    }

    if "selected_document" not in st.session_state:
        st.session_state.selected_document = None

    for document_name, document_file in documents.items():

        is_selected = (
            st.session_state.selected_document == document_file
        )

        button_label = (
            f"●  {document_name}"
            if is_selected
            else f"○  {document_name}"
        )

        if st.button(
            button_label,
            key=f"document_{document_file}",
            use_container_width=True,
        ):
            st.session_state.selected_document = document_file
            st.rerun()
'''

if old_block not in code:
    print("❌ Sidebar block was not found.")
else:
    code = code.replace(old_block, new_block)

    APP_PATH.write_text(code, encoding="utf-8")

    print("=" * 60)
    print("✅ SIDEBAR DOCUMENT SELECTION ADDED")
    print("=" * 60)
    print()
    print("Documents are now selectable:")
    print("✅ Leave Policy")
    print("✅ Attendance Policy")
    print("✅ Work From Home Policy")
    print("✅ Employee Handbook")
    print("✅ IT Security Policy")
    print("✅ Acceptable Use Policy")
    print()
    print("File updated:")
    print(APP_PATH)

✅ SIDEBAR DOCUMENT SELECTION ADDED

Documents are now selectable:
✅ Leave Policy
✅ Attendance Policy
✅ Work From Home Policy
✅ Employee Handbook
✅ IT Security Policy
✅ Acceptable Use Policy

File updated:
/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py


In [47]:
from pathlib import Path

APP_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py"
)

code = APP_PATH.read_text(encoding="utf-8")

old_code = """        result = answer_question(question)
"""

new_code = """        selected_document = st.session_state.get(
            "selected_document"
        )

        result = answer_question(
            question,
            document_name=selected_document
        )
"""

if old_code not in code:
    print("❌ Question handler code was not found.")
else:
    code = code.replace(old_code, new_code)

    APP_PATH.write_text(code, encoding="utf-8")

    print("=" * 60)
    print("✅ DOCUMENT-AWARE RAG CONNECTED")
    print("=" * 60)
    print()
    print("Question handler now uses:")
    print("• Selected document when one is chosen")
    print("• Entire knowledge base when no document is selected")
    print()
    print("File updated:")
    print(APP_PATH)

✅ DOCUMENT-AWARE RAG CONNECTED

Question handler now uses:
• Selected document when one is chosen
• Entire knowledge base when no document is selected

File updated:
/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py


In [48]:
import ast
from pathlib import Path

APP_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py"
)

print("=" * 60)
print("🔍 VALIDATING STREAMLIT APP")
print("=" * 60)
print()

try:
    code = APP_PATH.read_text(encoding="utf-8")

    # Check Python syntax
    ast.parse(code)

    print("✅ Python syntax: PASSED")
    print("✅ app.py can be parsed successfully")
    print()
    print("File:")
    print(APP_PATH)
    print()
    print("=" * 60)
    print("🎉 APP VALIDATION PASSED")
    print("=" * 60)

except SyntaxError as e:
    print("❌ SYNTAX ERROR")
    print(f"Line: {e.lineno}")
    print(f"Error: {e.msg}")
    print()
    print("Please send me the complete output.")

except Exception as e:
    print("❌ VALIDATION ERROR")
    print(type(e).__name__, ":", e)

🔍 VALIDATING STREAMLIT APP

✅ Python syntax: PASSED
✅ app.py can be parsed successfully

File:
/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py

🎉 APP VALIDATION PASSED


In [49]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant"
)

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rag_engine import answer_question

print("=" * 60)
print("🧪 TESTING DOCUMENT-AWARE RAG")
print("=" * 60)
print()

# Simulate selecting Leave Policy from the sidebar
selected_document = "Leave_Policy.docx"

question = "How many annual leave days are provided?"

print("Selected document:", selected_document)
print("Question:", question)
print()
print("Generating answer...")
print()

result = answer_question(
    question,
    document_name=selected_document
)

print("Answer:")
print(result["answer"])
print()

print("Sources:")
for source in result["sources"]:
    print(
        f"- {source['document']} | "
        f"Chunk: {source['chunk_id']} | "
        f"Score: {source['score']:.4f}"
    )

print()
print("Retrieved count:", result["retrieved_count"])
print()

if (
    result["retrieved_count"] > 0
    and all(
        source["document"] == selected_document
        for source in result["sources"]
    )
):
    print("=" * 60)
    print("✅ DOCUMENT-AWARE RAG TEST PASSED")
    print("=" * 60)
else:
    print("=" * 60)
    print("❌ DOCUMENT-AWARE RAG TEST FAILED")
    print("=" * 60)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded: all-MiniLM-L6-v2
Loading FAISS index...
FAISS index loaded!
Vectors: 20
Dimension: 384
Loading chunks...
Chunks loaded: 20
🧪 TESTING DOCUMENT-AWARE RAG

Selected document: Leave_Policy.docx
Question: How many annual leave days are provided?

Generating answer...

Answer:
Full-time employees are entitled to 20 days of annual leave per calendar year, as stated in **Leave_Policy.docx**.

Sources:
- Leave_Policy.docx | Chunk: Leave_Policy.docx_chunk_0010 | Score: 0.6500
- Leave_Policy.docx | Chunk: Leave_Policy.docx_chunk_0011 | Score: 0.4426
- Leave_Policy.docx | Chunk: Leave_Policy.docx_chunk_0012 | Score: 0.3491

Retrieved count: 3

✅ DOCUMENT-AWARE RAG TEST PASSED


In [51]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 44.8 MB/s eta 0:00:00


In [52]:
import shutil

streamlit_path = shutil.which("streamlit")

if streamlit_path:
    print("✅ Streamlit installed")
    print("Path:", streamlit_path)
else:
    print("❌ Streamlit is still not available")

✅ Streamlit installed
Path: /usr/local/bin/streamlit


In [53]:
import subprocess
import time
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant"
)

APP_PATH = PROJECT_ROOT / "app" / "app.py"

print("=" * 60)
print("🚀 STARTING TECHNOVA RAG ASSISTANT")
print("=" * 60)
print()
print("App:", APP_PATH)
print()

process = subprocess.Popen(
    [
        "/usr/local/bin/streamlit",
        "run",
        str(APP_PATH),
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(5)

print("✅ Streamlit process started")
print("Process ID:", process.pid)
print()
print("=" * 60)
print("🌐 STREAMLIT SERVER")
print("=" * 60)
print("Port: 8501")
print()
print("Next: expose port 8501 and open the app.")

🚀 STARTING TECHNOVA RAG ASSISTANT

App: /content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py

✅ Streamlit process started
Process ID: 12445

🌐 STREAMLIT SERVER
Port: 8501

Next: expose port 8501 and open the app.


In [57]:
import time

print("=" * 60)
print("🔍 CHECKING STREAMLIT SERVER LOGS")
print("=" * 60)
print()

time.sleep(2)

if process.poll() is None:
    print("✅ Streamlit process is still running")
else:
    print("❌ Streamlit process stopped")
    print("Exit code:", process.returncode)

print()
print("Server output:")
print("-" * 60)

# Read available output without blocking
if process.stdout:
    while True:
        line = process.stdout.readline()
        if not line:
            break
        print(line.rstrip())

print("-" * 60)

🔍 CHECKING STREAMLIT SERVER LOGS

❌ Streamlit process stopped
Exit code: 0

Server output:
------------------------------------------------------------
------------------------------------------------------------


In [59]:
process.terminate()
process.wait()

print("✅ Previous Streamlit process stopped")

✅ Previous Streamlit process stopped


In [60]:
import subprocess
import time
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant"
)

APP_PATH = PROJECT_ROOT / "app" / "app.py"
LOG_PATH = PROJECT_ROOT / "streamlit.log"

if LOG_PATH.exists():
    LOG_PATH.unlink()

cmd = [
    "python",
    "-m",
    "streamlit",
    "run",
    str(APP_PATH),
    "--server.port=8501",
    "--server.address=0.0.0.0",
    "--server.headless=true",
    "--server.enableCORS=false",
    "--server.enableXsrfProtection=false",
]

with open(LOG_PATH, "w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )

time.sleep(8)

print("=" * 60)
print("🚀 STREAMLIT STARTED FOR COLAB")
print("=" * 60)
print()
print("Process ID:", process.pid)
print("Running:", process.poll() is None)
print()

print("=" * 60)
print("📋 SERVER LOG")
print("=" * 60)
print()

print(LOG_PATH.read_text(encoding="utf-8"))

🚀 STREAMLIT STARTED FOR COLAB

Process ID: 14059
Running: True

📋 SERVER LOG



2026-09-04 09:30:51.351 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.229.49.180:8501




In [70]:
# ============================================================
# TECHNOVA AI
# DOCUMENT NAVIGATION + DEDICATED DOCUMENT PAGES
# + SOURCE ATTRIBUTION
# ============================================================

from pathlib import Path
import ast

APP_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py"
)


# ============================================================
# READ EXISTING APP
# ============================================================

app_code = APP_PATH.read_text(encoding="utf-8")


# ============================================================
# FIND EXISTING SESSION STATE SECTION
#
# Everything before this section is preserved.
# This includes:
# - imports
# - Streamlit configuration
# - CSS
# - existing visual design
# ============================================================

marker = (
    "# ============================================================\n"
    "# SESSION STATE"
)

if marker not in app_code:
    raise RuntimeError(
        "❌ Could not locate the SESSION STATE section."
    )

header = app_code.split(marker)[0]


# ============================================================
# NEW APPLICATION LOGIC
# ============================================================

logic = r'''
# ============================================================
# SESSION STATE
# ============================================================

if "messages" not in st.session_state:
    st.session_state.messages = []

if "selected_document" not in st.session_state:
    st.session_state.selected_document = None


# ============================================================
# DOCUMENT CONFIGURATION
# ============================================================

DOCUMENTS = {

    "leave": {
        "name": "Leave Policy",
        "file": "Leave_Policy.docx",
        "icon": "📅",
        "description":
            "Guidelines covering employee leave, eligibility, "
            "annual leave entitlement, and leave procedures.",
    },

    "attendance": {
        "name": "Attendance Policy",
        "file": "Attendance_Policy.docx",
        "icon": "🕐",
        "description":
            "Guidelines covering employee attendance, working "
            "hours, punctuality, and absence.",
    },

    "wfh": {
        "name": "Work From Home Policy",
        "file": "Work_From_Home_Policy.docx",
        "icon": "🏠",
        "description":
            "Guidelines for employees working remotely, including "
            "eligibility, approvals, and remote-work expectations.",
    },

    "handbook": {
        "name": "Employee Handbook",
        "file": "Employee_Handbook.docx",
        "icon": "👤",
        "description":
            "General information about TechNova Solutions, "
            "employee responsibilities, workplace practices, "
            "and organizational guidelines.",
    },

    "security": {
        "name": "IT Security Policy",
        "file": "IT_Security_Policy.docx",
        "icon": "🔐",
        "description":
            "Rules for protecting company systems, accounts, "
            "devices, information, and organizational data.",
    },

    "acceptable": {
        "name": "Acceptable Use Policy",
        "file": "Acceptable_Use_Policy.docx",
        "icon": "🛡️",
        "description":
            "Rules governing appropriate use of company systems, "
            "software, internet access, and network resources.",
    },

    "travel": {
        "name": "Travel Policy",
        "file": "Travel_Policy.docx",
        "icon": "✈️",
        "description":
            "Guidelines for official business travel, approvals, "
            "expenses, and travel procedures.",
    },

    "procurement": {
        "name": "Procurement SOP",
        "file": "Procurement_SOP.docx",
        "icon": "🛒",
        "description":
            "Standard operating procedures for organizational "
            "procurement and purchasing activities.",
    },
}


# ============================================================
# DOCUMENT DIRECTORY
# ============================================================

DOCUMENTS_DIR = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/Data/documents"
)


# ============================================================
# LOAD ACTUAL DOCX CONTENT
# ============================================================

def load_document_content(filename):
    """
    Load the actual content from the selected DOCX document.

    The document page displays real company document content
    instead of manually fabricated information.
    """

    from docx import Document

    file_path = DOCUMENTS_DIR / filename

    if not file_path.exists():
        return []

    document = Document(file_path)

    paragraphs = []

    for paragraph in document.paragraphs:

        text = paragraph.text.strip()

        if text:
            paragraphs.append(text)

    return paragraphs


# ============================================================
# FIND SELECTED DOCUMENT
# ============================================================

selected_file = st.session_state.selected_document

selected_info = None

for document_id, document in DOCUMENTS.items():

    if document["file"] == selected_file:

        selected_info = document

        break


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    # --------------------------------------------------------
    # BRAND
    # --------------------------------------------------------

    st.html("""
    <div style="
        display:flex;
        align-items:center;
        gap:14px;
    ">

        <div style="
            width:55px;
            height:55px;
            border-radius:17px;
            display:flex;
            align-items:center;
            justify-content:center;
            font-size:27px;
            background:linear-gradient(
                135deg,
                #6366f1,
                #06b6d4
            );
        ">
            🤖
        </div>

        <div>

            <div class="sidebar-title">
                TechNova AI
            </div>

            <div class="sidebar-subtitle">
                Enterprise Knowledge Assistant
            </div>

        </div>

    </div>
    """)


    # --------------------------------------------------------
    # SYSTEM STATUS
    # --------------------------------------------------------

    st.html("""
    <div class="status-card">

        <div class="status-label">
            SYSTEM STATUS
        </div>

        <div class="status-value">
            🟢 AI system online
        </div>

    </div>
    """)


    # --------------------------------------------------------
    # KNOWLEDGE BASE
    # --------------------------------------------------------

    st.html("""
    <div class="sidebar-heading">
        Knowledge Base
    </div>
    """)


    # --------------------------------------------------------
    # DOCUMENT BUTTONS
    # --------------------------------------------------------

    for document_id, document in DOCUMENTS.items():

        is_selected = (
            st.session_state.selected_document
            == document["file"]
        )

        if is_selected:

            label = (
                f"{document['icon']}  "
                f"{document['name']}  ✓"
            )

        else:

            label = (
                f"{document['icon']}  "
                f"{document['name']}"
            )


        if st.button(
            label,
            key=f"document_button_{document_id}",
            use_container_width=True,
        ):

            # Store selected document
            st.session_state.selected_document = (
                document["file"]
            )

            # Start fresh conversation
            st.session_state.messages = []

            # Reload application
            st.rerun()


    # --------------------------------------------------------
    # SIDEBAR BACK TO ASSISTANT
    # --------------------------------------------------------

    if selected_info is not None:

        st.markdown("---")

        if st.button(
            "← Back to Assistant",
            key="sidebar_back_to_assistant",
            use_container_width=True,
        ):

            # Remove selected document
            st.session_state.selected_document = None

            # Clear document-specific conversation
            st.session_state.messages = []

            # Return to main assistant
            st.rerun()


    # --------------------------------------------------------
    # ASSISTANT CAPABILITIES
    # --------------------------------------------------------

    st.html("""
    <div class="sidebar-heading">
        Assistant Capabilities
    </div>
    """)

    st.markdown(
        "🔎 Semantic document search"
    )

    st.markdown(
        "📚 Source attribution"
    )

    st.markdown(
        "🛡️ Grounded responses"
    )

    st.markdown(
        "💬 Conversation history"
    )


    # --------------------------------------------------------
    # CLEAR CONVERSATION
    # --------------------------------------------------------

    st.markdown("---")

    if st.button(
        "🗑️ Clear Conversation",
        use_container_width=True,
    ):

        st.session_state.messages = []

        st.rerun()


# ============================================================
# DOCUMENT INFORMATION PAGE
# ============================================================

if selected_info is not None:

    # ========================================================
    # PAGE-LEVEL BACK BUTTON
    # ========================================================

    if st.button(
        "← Back to Assistant",
        key="page_back_to_assistant",
        use_container_width=False,
    ):

        # Remove selected document
        st.session_state.selected_document = None

        # Clear document-specific chat
        st.session_state.messages = []

        # Return to main assistant
        st.rerun()


    # ========================================================
    # DOCUMENT HERO
    # ========================================================

    st.html(
        f"""
        <div class="hero">

            <div class="hero-badge">
                📚 KNOWLEDGE BASE DOCUMENT
            </div>

            <div class="hero-title">
                {selected_info["icon"]}
                {selected_info["name"]}
            </div>

            <div class="hero-text">
                {selected_info["description"]}
            </div>

        </div>
        """
    )


    # ========================================================
    # DOCUMENT METADATA
    # ========================================================

    st.markdown(
        "### 📄 Document Information"
    )

    col1, col2 = st.columns(2)

    with col1:

        st.markdown(
            f"""
            **Document**

            `{selected_info["file"]}`
            """
        )

    with col2:

        st.markdown(
            """
            **Organization**

            `TechNova Solutions`
            """
        )


    # ========================================================
    # ACTUAL DOCUMENT CONTENT
    # ========================================================

    st.markdown(
        "### 📖 Detailed Policy Information"
    )

    document_content = load_document_content(
        selected_info["file"]
    )


    if document_content:

        for paragraph in document_content:

            # Detect common section headings
            if (
                paragraph[:2].isdigit()
                or paragraph.startswith(
                    (
                        "Introduction",
                        "Purpose",
                        "Scope",
                        "Eligibility",
                        "Responsibilities",
                        "Procedure",
                        "Policy",
                        "Guidelines",
                        "Exceptions",
                    )
                )
            ):

                st.markdown(
                    f"#### {paragraph}"
                )

            else:

                st.markdown(paragraph)

    else:

        st.warning(
            "⚠️ The document could not be loaded."
        )


    # ========================================================
    # DOCUMENT AI ASSISTANT
    # ========================================================

    st.markdown("---")

    st.markdown(
        f"### 🤖 Ask AI about {selected_info['name']}"
    )

    st.caption(
        "Questions are answered using this document "
        "as the selected knowledge source."
    )


    # ========================================================
    # DOCUMENT CHAT HISTORY
    # ========================================================

    for message in st.session_state.messages:

        role = message["role"]

        content = message["content"]


        # ----------------------------------------------------
        # USER MESSAGE
        # ----------------------------------------------------

        if role == "user":

            st.html(
                f"""
                <div class="user-message">

                    <div class="message-label">
                        You
                    </div>

                    <div>
                        {content}
                    </div>

                </div>
                """
            )


        # ----------------------------------------------------
        # ASSISTANT MESSAGE
        # ----------------------------------------------------

        else:

            st.html(
                f"""
                <div class="assistant-message">

                    <div class="message-label">
                        🤖 TechNova AI
                    </div>

                    <div>
                        {content}
                    </div>

                </div>
                """
            )


            # =================================================
            # DISPLAY SOURCES
            # =================================================

            sources = message.get(
                "sources",
                []
            )

            if sources:

                st.markdown(
                    "##### 📚 Sources"
                )

                for source_index, source in enumerate(
                    sources,
                    start=1
                ):

                    document_name = source.get(
                        "document",
                        "Unknown document"
                    )

                    chunk_id = source.get(
                        "chunk_id",
                        "Unknown chunk"
                    )

                    score = source.get(
                        "score",
                        0
                    )

                    st.markdown(
                        f"""
                        **{source_index}. 📄 {document_name}**

                        `Chunk: {chunk_id}`

                        `Relevance: {score:.4f}`
                        """
                    )


    # ========================================================
    # DOCUMENT QUESTION
    # ========================================================

    question = st.chat_input(
        f"Ask anything about {selected_info['name']}..."
    )


    if question:

        # Add user question
        st.session_state.messages.append(
            {
                "role": "user",
                "content": question,
            }
        )


        # ----------------------------------------------------
        # DOCUMENT-SPECIFIC RAG
        # ----------------------------------------------------

        with st.spinner(
            "🔎 Searching this document..."
        ):

            result = answer_question(
                question,
                document_name=selected_info["file"],
            )


        # ----------------------------------------------------
        # GET ANSWER
        # ----------------------------------------------------

        answer = result.get(
            "answer",
            "I couldn't generate an answer.",
        )


        # ----------------------------------------------------
        # GET SOURCES
        # ----------------------------------------------------

        sources = result.get(
            "sources",
            [],
        )


        # ----------------------------------------------------
        # STORE ASSISTANT RESPONSE
        # ----------------------------------------------------

        st.session_state.messages.append(
            {
                "role": "assistant",
                "content": answer,
                "sources": sources,
                "retrieved_count":
                    result.get(
                        "retrieved_count",
                        0,
                    ),
            }
        )


        # Reload page
        st.rerun()


# ============================================================
# MAIN ASSISTANT PAGE
# ============================================================

else:

    # ========================================================
    # MAIN HERO
    # ========================================================

    st.html(
        """
        <div class="hero">

            <div class="hero-badge">
                ● Secure Knowledge Assistant
            </div>

            <div class="hero-title">
                Ask your company knowledge.
            </div>

            <div class="hero-text">
                Search internal policies, employee documentation,
                and organizational knowledge using natural language.
                Answers are grounded in the retrieved company documents.
            </div>

        </div>
        """
    )


    # ========================================================
    # EXAMPLE QUESTIONS
    # ========================================================

    st.markdown(
        '<div class="sidebar-heading">✨ Try asking</div>',
        unsafe_allow_html=True,
    )


    example_questions = [
        "How many annual leave days do employees get?",
        "What are the core working hours?",
        "How many consecutive days can employees work from home?",
    ]


    cols = st.columns(3)


    for i, example in enumerate(example_questions):

        with cols[i]:

            if st.button(
                example,
                key=f"example_question_{i}",
                use_container_width=True,
            ):

                st.session_state.pending_question = example

                st.rerun()


    # ========================================================
    # MAIN CHAT HISTORY
    # ========================================================

    for message in st.session_state.messages:

        role = message["role"]

        content = message["content"]


        # ----------------------------------------------------
        # USER MESSAGE
        # ----------------------------------------------------

        if role == "user":

            st.html(
                f"""
                <div class="user-message">

                    <div class="message-label">
                        You
                    </div>

                    <div>
                        {content}
                    </div>

                </div>
                """
            )


        # ----------------------------------------------------
        # ASSISTANT MESSAGE
        # ----------------------------------------------------

        else:

            st.html(
                f"""
                <div class="assistant-message">

                    <div class="message-label">
                        🤖 TechNova AI
                    </div>

                    <div>
                        {content}
                    </div>

                </div>
                """
            )


            # =================================================
            # DISPLAY SOURCES
            # =================================================

            sources = message.get(
                "sources",
                []
            )

            if sources:

                st.markdown(
                    "##### 📚 Sources"
                )

                for source_index, source in enumerate(
                    sources,
                    start=1
                ):

                    document_name = source.get(
                        "document",
                        "Unknown document"
                    )

                    chunk_id = source.get(
                        "chunk_id",
                        "Unknown chunk"
                    )

                    score = source.get(
                        "score",
                        0
                    )

                    st.markdown(
                        f"""
                        **{source_index}. 📄 {document_name}**

                        `Chunk: {chunk_id}`

                        `Relevance: {score:.4f}`
                        """
                    )


    # ========================================================
    # PENDING EXAMPLE QUESTION
    # ========================================================

    pending_question = st.session_state.pop(
        "pending_question",
        None,
    )


    # ========================================================
    # CHAT INPUT
    # ========================================================

    question = st.chat_input(
        "Ask anything about company policies..."
    )


    if pending_question:

        question = pending_question


    # ========================================================
    # PROCESS QUESTION
    # ========================================================

    if question:

        # Add user question
        st.session_state.messages.append(
            {
                "role": "user",
                "content": question,
            }
        )


        # ----------------------------------------------------
        # GENERAL RAG SEARCH
        # ----------------------------------------------------

        with st.spinner(
            "🔎 Searching company knowledge..."
        ):

            result = answer_question(
                question
            )


        # ----------------------------------------------------
        # GET ANSWER
        # ----------------------------------------------------

        answer = result.get(
            "answer",
            "I couldn't generate an answer.",
        )


        # ----------------------------------------------------
        # GET SOURCES
        # ----------------------------------------------------

        sources = result.get(
            "sources",
            [],
        )


        # ----------------------------------------------------
        # STORE ASSISTANT RESPONSE
        # ----------------------------------------------------

        st.session_state.messages.append(
            {
                "role": "assistant",
                "content": answer,
                "sources": sources,
                "retrieved_count":
                    result.get(
                        "retrieved_count",
                        0,
                    ),
            }
        )


        # Reload page
        st.rerun()


# ============================================================
# FOOTER
# ============================================================

st.markdown(
    """
    <div style="
        text-align:center;
        margin-top:3rem;
        padding-top:1.2rem;
        border-top:1px solid rgba(255,255,255,0.06);
        color:#475569;
        font-size:11px;
    ">
        TechNova AI • Enterprise Document Intelligence
        • Retrieval-Augmented Generation
    </div>
    """,
    unsafe_allow_html=True,
)


# ============================================================
# END OF APPLICATION
# ============================================================
'''


# ============================================================
# WRITE UPDATED APP
# ============================================================

updated_app = header + logic

APP_PATH.write_text(
    updated_app,
    encoding="utf-8",
)


# ============================================================
# SYNTAX VALIDATION
# ============================================================

ast.parse(updated_app)


# ============================================================
# SUCCESS OUTPUT
# ============================================================

print("=" * 75)
print("✅ TECHNOVA AI APP UPDATED SUCCESSFULLY")
print("=" * 75)

print()

print("📚 Knowledge Base Documents:")

print("📅  Leave Policy")
print("🕐  Attendance Policy")
print("🏠  Work From Home Policy")
print("👤  Employee Handbook")
print("🔐  IT Security Policy")
print("🛡️  Acceptable Use Policy")
print("✈️  Travel Policy")
print("🛒  Procurement SOP")

print()

print("Document page features:")

print("✔ Dedicated document information page")
print("✔ Actual DOCX content loaded")
print("✔ Document metadata")
print("✔ Document-specific AI chat")
print("✔ Page-level ← Back to Assistant")
print("✔ Sidebar ← Back to Assistant")
print("✔ 8 documents supported")

print()

print("Chat rendering:")

print("✔ User messages use st.html()")
print("✔ AI messages use st.html()")
print("✔ HTML <div> tags will not appear as text")
print("✔ Main assistant chat fixed")
print("✔ Document-specific chat fixed")

print()

print("Source attribution:")

print("✔ Sources preserved from RAG engine")
print("✔ Source document displayed")
print("✔ Chunk ID displayed")
print("✔ Relevance score displayed")
print("✔ Sources shown on main assistant")
print("✔ Sources shown on document pages")

print()

print("Navigation flow:")

print("🏠 Main Assistant")
print("      ↓")
print("📚 Select Document")
print("      ↓")
print("📄 Dedicated Document Page")
print("      ↓")
print("🤖 Document AI")
print("      ↓")
print("📚 Sources")
print("      ↓")
print("← Back to Assistant")
print("      ↓")
print("🏠 Main Assistant")

print()

print("System safety:")

print("✔ RAG engine was NOT modified")
print("✔ FAISS index was NOT modified")
print("✔ Gemini configuration was NOT modified")
print("✔ Python syntax validation passed")

print()

print("File:")
print(APP_PATH)

print()
print("=" * 75)

✅ TECHNOVA AI APP UPDATED SUCCESSFULLY

📚 Knowledge Base Documents:
📅  Leave Policy
🕐  Attendance Policy
🏠  Work From Home Policy
👤  Employee Handbook
🔐  IT Security Policy
🛡️  Acceptable Use Policy
✈️  Travel Policy
🛒  Procurement SOP

Document page features:
✔ Dedicated document information page
✔ Actual DOCX content loaded
✔ Document metadata
✔ Document-specific AI chat
✔ Page-level ← Back to Assistant
✔ Sidebar ← Back to Assistant
✔ 8 documents supported

Chat rendering:
✔ User messages use st.html()
✔ AI messages use st.html()
✔ HTML <div> tags will not appear as text
✔ Main assistant chat fixed
✔ Document-specific chat fixed

Source attribution:
✔ Sources preserved from RAG engine
✔ Source document displayed
✔ Chunk ID displayed
✔ Relevance score displayed
✔ Sources shown on main assistant
✔ Sources shown on document pages

Navigation flow:
🏠 Main Assistant
      ↓
📚 Select Document
      ↓
📄 Dedicated Document Page
      ↓
🤖 Document AI
      ↓
📚 Sources
      ↓
← Back to Assist

In [55]:
from google.colab import output

print("=" * 60)
print("🌐 CREATING STREAMLIT ACCESS URL")
print("=" * 60)
print()

url = output.serve_kernel_port_as_window(8501)

print()
print("=" * 60)
print("✅ STREAMLIT URL CREATED")
print("=" * 60)
print()
print(url)

🌐 CREATING STREAMLIT ACCESS URL

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>


✅ STREAMLIT URL CREATED

None


In [71]:
# ============================================================
# STEP 1 — ADD SOURCE CARDS TO THE STREAMLIT APP
# ============================================================

from pathlib import Path

APP_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py"
)

# Read the current working app
app_code = APP_PATH.read_text(encoding="utf-8")

# ------------------------------------------------------------
# Helper function: render source cards
# ------------------------------------------------------------
source_renderer = '''
def render_sources(message):
    """Display retrieved document sources below an AI response."""

    sources = message.get("sources", [])
    retrieved_count = message.get("retrieved_count", 0)

    if not sources:
        return

    st.html("""
    <div style="
        margin-top: 12px;
        margin-bottom: 8px;
        font-size: 15px;
        font-weight: 700;
    ">
        📚 Sources
    </div>
    """)

    for source in sources:
        document = source.get("document", "Unknown document")
        chunk_id = source.get("chunk_id", "N/A")
        score = float(source.get("score", 0))

        st.html(f"""
        <div class="source-card">
            <div class="source-item">
                <div>
                    <strong>📄 {document}</strong>
                </div>
                <div style="margin-top: 5px;">
                    <span>Chunk: {chunk_id}</span>
                </div>
                <div style="margin-top: 3px;">
                    <span>Relevance: {score:.2f}</span>
                </div>
            </div>
        </div>
        """)

    st.html(f"""
    <div style="
        margin-top: 5px;
        margin-bottom: 15px;
        font-size: 12px;
        opacity: 0.65;
    ">
        Retrieved passages: {retrieved_count}
    </div>
    """)
'''

# ------------------------------------------------------------
# Insert helper before SESSION STATE
# ------------------------------------------------------------
if "def render_sources(message):" not in app_code:

    marker = "# SESSION STATE"

    if marker in app_code:
        app_code = app_code.replace(
            marker,
            source_renderer + "\n\n" + marker,
            1
        )
    else:
        raise RuntimeError(
            "Could not find '# SESSION STATE' in app.py"
        )

# ------------------------------------------------------------
# Add source rendering after assistant messages
# ------------------------------------------------------------
old_assistant_pattern = '''    else:
        st.html(f"""
        <div class="assistant-message">
            <div class="message-label">🤖 AI Assistant</div>
            <div>{content}</div>
        </div>
        """)
'''

new_assistant_pattern = '''    else:
        st.html(f"""
        <div class="assistant-message">
            <div class="message-label">🤖 AI Assistant</div>
            <div>{content}</div>
        </div>
        """)

        # Display retrieved sources underneath the answer
        render_sources(message)
'''

# Replace all matching assistant message blocks
if old_assistant_pattern in app_code:
    app_code = app_code.replace(
        old_assistant_pattern,
        new_assistant_pattern
    )
else:
    print("⚠️ Standard assistant message block was not found.")
    print("The app may already contain a modified message renderer.")

# ------------------------------------------------------------
# Save updated app
# ------------------------------------------------------------
APP_PATH.write_text(app_code, encoding="utf-8")

print("=" * 60)
print("✅ SOURCE CARDS ADDED SUCCESSFULLY")
print("=" * 60)
print()
print("Updated file:")
print(APP_PATH)
print()
print("Source information displayed:")
print("  📄 Document name")
print("  🔹 Chunk ID")
print("  📊 Relevance score")
print("  📚 Retrieved passage count")

⚠️ Standard assistant message block was not found.
The app may already contain a modified message renderer.
✅ SOURCE CARDS ADDED SUCCESSFULLY

Updated file:
/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py

Source information displayed:
  📄 Document name
  🔹 Chunk ID
  📊 Relevance score
  📚 Retrieved passage count


In [72]:
# Check that app.py has no syntax errors

import ast

APP_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py"
)

code = APP_PATH.read_text(encoding="utf-8")

ast.parse(code)

print("✅ app.py syntax is valid")
print("✅ Source renderer is present:", "def render_sources(message):" in code)

✅ app.py syntax is valid
✅ Source renderer is present: True


In [73]:
# ============================================================
# INSPECT CURRENT ASSISTANT MESSAGE RENDERING
# ============================================================

from pathlib import Path

APP_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py"
)

code = APP_PATH.read_text(encoding="utf-8")
lines = code.splitlines()

print("=" * 70)
print("SEARCHING FOR ASSISTANT MESSAGE RENDERING")
print("=" * 70)

matches = []

for i, line in enumerate(lines, start=1):
    if (
        "assistant-message" in line
        or "message[\"role\"]" in line
        or "message['role']" in line
        or "role = message" in line
    ):
        matches.append(i)

print("Matching lines:", matches)
print()

# Show context around every relevant location
for line_number in matches:
    start = max(1, line_number - 8)
    end = min(len(lines), line_number + 15)

    print("-" * 70)
    print(f"LINES {start} - {end}")
    print("-" * 70)

    for i in range(start, end + 1):
        print(f"{i:4}: {lines[i-1]}")

print()
print("=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

SEARCHING FOR ASSISTANT MESSAGE RENDERING
Matching lines: [321, 1019, 1055, 1266, 1302]

----------------------------------------------------------------------
LINES 313 - 336
----------------------------------------------------------------------
 313:         margin: 15px 0 10px auto;
 314: 
 315:         max-width: 88%;
 316: 
 317:         color: #e2e8f0;
 318:         line-height: 1.6;
 319:     }
 320: 
 321:     .assistant-message {
 322:         background:
 323:             linear-gradient(
 324:                 135deg,
 325:                 rgba(30,41,59,0.85),
 326:                 rgba(15,23,42,0.78)
 327:             );
 328: 
 329:         border: 1px solid rgba(148,163,184,0.10);
 330: 
 331:         border-radius: 5px 18px 18px 18px;
 332: 
 333:         padding: 18px 20px;
 334: 
 335:         margin: 8px auto 15px 0;
 336: 
----------------------------------------------------------------------
LINES 1011 - 1034
----------------------------------------------------------

In [74]:
# ============================================================
# STEP 2 — CONNECT SOURCE CARDS TO BOTH CHAT HISTORIES
# ============================================================

from pathlib import Path

APP_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py"
)

app_code = APP_PATH.read_text(encoding="utf-8")


# ============================================================
# 1. DOCUMENT CHAT — ADD SOURCE RENDERING
# ============================================================

old_document_chat = '''            st.html(
                f"""
                <div class="assistant-message">

                    <div class="message-label">
                        🤖 TechNova AI
                    </div>

                    <div>
                        {content}
                    </div>

                </div>
                """
            )


            # =================================================
'''

new_document_chat = '''            st.html(
                f"""
                <div class="assistant-message">

                    <div class="message-label">
                        🤖 TechNova AI
                    </div>

                    <div>
                        {content}
                    </div>

                </div>
                """
            )

            # ------------------------------------------------
            # DISPLAY SOURCES FOR THIS AI RESPONSE
            # ------------------------------------------------
            render_sources(message)


            # =================================================
'''

# Replace only the first occurrence
if old_document_chat in app_code:
    app_code = app_code.replace(
        old_document_chat,
        new_document_chat,
        1
    )
    print("✅ Document chat source rendering connected")
else:
    print("⚠️ Document chat block not found")


# ============================================================
# 2. MAIN CHAT — ADD SOURCE RENDERING
# ============================================================

old_main_chat = '''            st.html(
                f"""
                <div class="assistant-message">

                    <div class="message-label">
                        🤖 TechNova AI
                    </div>

                    <div>
                        {content}
                    </div>

                </div>
                """
            )


            # =================================================
'''

new_main_chat = '''            st.html(
                f"""
                <div class="assistant-message">

                    <div class="message-label">
                        🤖 TechNova AI
                    </div>

                    <div>
                        {content}
                    </div>

                </div>
                """
            )

            # ------------------------------------------------
            # DISPLAY SOURCES FOR THIS AI RESPONSE
            # ------------------------------------------------
            render_sources(message)


            # =================================================
'''

# Replace the remaining occurrence
if old_main_chat in app_code:
    app_code = app_code.replace(
        old_main_chat,
        new_main_chat,
        1
    )
    print("✅ Main chat source rendering connected")
else:
    print("⚠️ Main chat block not found")


# ============================================================
# 3. SAVE
# ============================================================

APP_PATH.write_text(app_code, encoding="utf-8")


# ============================================================
# 4. SYNTAX VALIDATION
# ============================================================

import ast

code = APP_PATH.read_text(encoding="utf-8")
ast.parse(code)


# ============================================================
# 5. FINAL CHECK
# ============================================================

source_calls = code.count("render_sources(message)")

print()
print("=" * 70)
print("✅ SOURCE CARD PATCH COMPLETE")
print("=" * 70)
print()
print("Source renderer calls found:", source_calls)
print("Expected:", 2)
print()
print("✅ app.py syntax is valid")
print("✅ Document chat connected")
print("✅ Main chat connected")
print()
print("🚀 Ready to test in Streamlit")

✅ Document chat source rendering connected
✅ Main chat source rendering connected

✅ SOURCE CARD PATCH COMPLETE

Source renderer calls found: 3
Expected: 2

✅ app.py syntax is valid
✅ Document chat connected
✅ Main chat connected

🚀 Ready to test in Streamlit


In [75]:
# ============================================================
# STEP 3 — RESTART STREAMLIT
# ============================================================

import subprocess
import time

# Stop existing Streamlit processes
subprocess.run(
    ["pkill", "-f", "streamlit"],
    capture_output=True
)

time.sleep(2)

# Start Streamlit again
APP_PATH = "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py"

cmd = [
    "python",
    "-m",
    "streamlit",
    "run",
    APP_PATH,
    "--server.port=8501",
    "--server.address=0.0.0.0",
    "--server.headless=true",
    "--server.enableCORS=false",
    "--server.enableXsrfProtection=false",
]

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("=" * 60)
print("🚀 STREAMLIT RESTARTED")
print("=" * 60)
print()
print("PID:", process.pid)
print("URL: http://localhost:8501")
print()
print("Open the Streamlit window from the previous step.")

🚀 STREAMLIT RESTARTED

PID: 29163
URL: http://localhost:8501

Open the Streamlit window from the previous step.


In [76]:
# ============================================================
# STEP 4 — CHECK STREAMLIT PROCESS AND ERROR LOG
# ============================================================

import subprocess
from pathlib import Path

APP_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py"
)

print("=" * 70)
print("🔍 CHECKING STREAMLIT")
print("=" * 70)

# Check whether Streamlit is currently running
result = subprocess.run(
    ["pgrep", "-af", "streamlit"],
    capture_output=True,
    text=True
)

if result.stdout.strip():
    print("🟢 Streamlit process found:")
    print(result.stdout)
else:
    print("🔴 No Streamlit process is currently running.")

print()
print("Checking app.py syntax...")

import ast

try:
    code = APP_PATH.read_text(encoding="utf-8")
    ast.parse(code)
    print("✅ app.py syntax is valid")
except Exception as e:
    print("❌ Syntax error:")
    print(e)

print()
print("=" * 70)
print("CHECK COMPLETE")
print("=" * 70)

🔍 CHECKING STREAMLIT
🟢 Streamlit process found:
29163 python3 -m streamlit run /content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py --server.port=8501 --server.address=0.0.0.0 --server.headless=true --server.enableCORS=false --server.enableXsrfProtection=false


Checking app.py syntax...
✅ app.py syntax is valid

CHECK COMPLETE


In [77]:
# ============================================================
# STEP 5 — OPEN STREAMLIT THROUGH COLAB
# ============================================================

from google.colab import output

print("🌐 Opening Streamlit through the Colab runtime...")
print("Port: 8501")

output.serve_kernel_port_as_window(8501)

🌐 Opening Streamlit through the Colab runtime...
Port: 8501
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [78]:
from pathlib import Path
import shutil

APP_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py"
)

BACKUP_PATH = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app_working_backup.py"
)

shutil.copy2(APP_PATH, BACKUP_PATH)

print("=" * 60)
print("✅ WORKING APP BACKUP CREATED")
print("=" * 60)
print()
print("Original:")
print(APP_PATH)
print()
print("Backup:")
print(BACKUP_PATH)

✅ WORKING APP BACKUP CREATED

Original:
/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py

Backup:
/content/drive/MyDrive/Enterprise_RAG_Assistant/app/app_working_backup.py


In [79]:
# ============================================================
# STEP 8 — FINAL PROJECT STRUCTURE CHECK
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant"
)

print("=" * 70)
print("🔍 ENTERPRISE RAG ASSISTANT — PROJECT CHECK")
print("=" * 70)

print("\n📁 Project root:")
print(PROJECT_ROOT)

# ------------------------------------------------------------
# Expected important files/folders
# ------------------------------------------------------------

required_items = {
    "src": PROJECT_ROOT / "src",
    "rag_engine.py": PROJECT_ROOT / "src" / "rag_engine.py",

    "app": PROJECT_ROOT / "app",
    "app.py": PROJECT_ROOT / "app" / "app.py",

    "vectorstore": PROJECT_ROOT / "vectorstore",
    "FAISS index": PROJECT_ROOT / "vectorstore" / "tech_nova.index",
    "Chunks": PROJECT_ROOT / "vectorstore" / "chunks.pkl",

    "documents": PROJECT_ROOT / "Data" / "documents",
}

# ------------------------------------------------------------
# Check every item
# ------------------------------------------------------------

print("\n📋 Required components:\n")

all_ok = True

for name, path in required_items.items():

    if path.exists():
        print(f"✅ {name}: FOUND")
    else:
        print(f"❌ {name}: MISSING")
        all_ok = False

# ------------------------------------------------------------
# Count documents
# ------------------------------------------------------------

documents_dir = PROJECT_ROOT / "Data" / "documents"

if documents_dir.exists():

    docx_files = sorted(documents_dir.glob("*.docx"))

    print("\n📄 DOCX documents found:", len(docx_files))

    for file in docx_files:
        print("   └──", file.name)

# ------------------------------------------------------------
# Vectorstore information
# ------------------------------------------------------------

index_path = PROJECT_ROOT / "vectorstore" / "tech_nova.index"
chunks_path = PROJECT_ROOT / "vectorstore" / "chunks.pkl"

print("\n🧠 Vector store:")

if index_path.exists():
    print("   ✅ FAISS index exists")
    print("   📦 Size:", round(index_path.stat().st_size / 1024, 2), "KB")
else:
    print("   ❌ FAISS index missing")
    all_ok = False

if chunks_path.exists():
    print("   ✅ chunks.pkl exists")
    print("   📦 Size:", round(chunks_path.stat().st_size / 1024, 2), "KB")
else:
    print("   ❌ chunks.pkl missing")
    all_ok = False

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("\n" + "=" * 70)

if all_ok:
    print("🎉 PROJECT STRUCTURE CHECK PASSED")
    print("✅ All critical project components are present")
else:
    print("⚠️ PROJECT STRUCTURE CHECK FOUND MISSING ITEMS")

print("=" * 70)

🔍 ENTERPRISE RAG ASSISTANT — PROJECT CHECK

📁 Project root:
/content/drive/MyDrive/Enterprise_RAG_Assistant

📋 Required components:

✅ src: FOUND
✅ rag_engine.py: FOUND
✅ app: FOUND
✅ app.py: FOUND
✅ vectorstore: FOUND
✅ FAISS index: FOUND
✅ Chunks: FOUND
✅ documents: FOUND

📄 DOCX documents found: 8
   └── Acceptable_Use_Policy.docx
   └── Attendance_Policy.docx
   └── Employee_Handbook.docx
   └── IT_Security_Policy.docx
   └── Leave_Policy.docx
   └── Procurement_SOP.docx
   └── Travel_Policy.docx
   └── Work_From_Home_Policy.docx

🧠 Vector store:
   ✅ FAISS index exists
   📦 Size: 30.04 KB
   ✅ chunks.pkl exists
   📦 Size: 10.86 KB

🎉 PROJECT STRUCTURE CHECK PASSED
✅ All critical project components are present


In [80]:
# ============================================================
# STEP 9 — CREATE REQUIREMENTS.TXT
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant"
)

requirements = """streamlit
google-genai
sentence-transformers
faiss-cpu
python-docx
numpy
"""

requirements_path = PROJECT_ROOT / "requirements.txt"

requirements_path.write_text(
    requirements,
    encoding="utf-8"
)

print("=" * 70)
print("✅ requirements.txt CREATED")
print("=" * 70)

print("\n📄 Location:")
print(requirements_path)

print("\n📦 Dependencies:")
print(requirements)

✅ requirements.txt CREATED

📄 Location:
/content/drive/MyDrive/Enterprise_RAG_Assistant/requirements.txt

📦 Dependencies:
streamlit
google-genai
sentence-transformers
faiss-cpu
python-docx
numpy



In [82]:
# ============================================================
# STEP 10 — SAFELY CREATE README.MD
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant"
)

README_PATH = PROJECT_ROOT / "README.md"

# ------------------------------------------------------------
# README CONTENT
# ------------------------------------------------------------

readme_lines = [
    "# 🧠 Enterprise RAG Assistant",
    "",
    "An AI-powered Enterprise Retrieval-Augmented Generation (RAG) Assistant "
    "designed to answer questions from internal organizational documents.",
    "",
    "The system retrieves relevant information from company documents and "
    "uses Google Gemini to generate grounded answers based only on retrieved "
    "company knowledge.",
    "",
    "---",
    "",
    "## 🏢 Organization",
    "",
    "**TechNova Solutions**",
    "",
    "The project uses internal company policies and standard operating "
    "procedures as its knowledge base.",
    "",
    "---",
    "",
    "## 🎯 Objective",
    "",
    "The objective is to build an intelligent enterprise assistant that "
    "allows employees to quickly search and understand internal company "
    "policies and procedures using natural-language questions.",
    "",
    "Instead of manually searching through multiple documents, users can "
    "ask questions directly to the AI assistant.",
    "",
    "---",
    "",
    "## ✨ Key Features",
    "",
    "- 🤖 AI-powered question answering",
    "- 📚 Retrieval-Augmented Generation (RAG)",
    "- 🔎 Semantic document search",
    "- 🧠 Sentence Transformer embeddings",
    "- ⚡ FAISS vector similarity search",
    "- 💬 Google Gemini response generation",
    "- 📄 Multiple DOCX company documents",
    "- 🗂️ Document-specific pages",
    "- 📚 Source cards with retrieved documents",
    "- 📊 Relevance scores",
    "- 🎨 Streamlit web interface",
    "",
    "---",
    "",
    "## 🏗️ System Architecture",
    "",
    "```text",
    "Company Documents (.docx)",
    "        ↓",
    "Document Loading",
    "        ↓",
    "Text Processing / Chunking",
    "        ↓",
    "Sentence Transformer",
    "all-MiniLM-L6-v2",
    "        ↓",
    "FAISS Vector Index",
    "        ↓",
    "User Question",
    "        ↓",
    "Query Embedding",
    "        ↓",
    "FAISS Similarity Search",
    "        ↓",
    "Relevant Context + Sources",
    "        ↓",
    "Google Gemini",
    "        ↓",
    "Grounded AI Answer + Sources",
    "```",
    "",
    "---",
    "",
    "## 🔄 RAG Pipeline",
    "",
    "### 1. Document Collection",
    "",
    "Company policies and SOP documents are stored as DOCX files.",
    "",
    "### 2. Document Loading",
    "",
    "Documents are loaded using `python-docx`.",
    "",
    "### 3. Text Processing",
    "",
    "Text is extracted and divided into searchable chunks.",
    "",
    "### 4. Embedding Generation",
    "",
    "Each chunk is converted into a numerical vector using:",
    "",
    "`all-MiniLM-L6-v2`",
    "",
    "### 5. Vector Storage",
    "",
    "The embeddings are stored in a FAISS vector index.",
    "",
    "The embedding dimension is **384**.",
    "",
    "### 6. User Query",
    "",
    "The employee enters a natural-language question.",
    "",
    "Example:",
    "",
    "`How many annual leave days are provided?`",
    "",
    "### 7. Query Embedding",
    "",
    "The question is converted into the same embedding space as the "
    "document chunks.",
    "",
    "### 8. Similarity Search",
    "",
    "FAISS searches for semantically relevant document chunks.",
    "",
    "### 9. Context Retrieval",
    "",
    "The retrieved passages are provided to the language model as context.",
    "",
    "### 10. Answer Generation",
    "",
    "Google Gemini generates the final response using the retrieved "
    "company information.",
    "",
    "### 11. Source Display",
    "",
    "The application displays:",
    "",
    "- Document name",
    "- Chunk ID",
    "- Relevance score",
    "- Retrieved passage count",
    "",
    "---",
    "",
    "## 🧰 Technologies Used",
    "",
    "| Component | Technology |",
    "|---|---|",
    "| Programming Language | Python |",
    "| Frontend | Streamlit |",
    "| LLM | Google Gemini |",
    "| Embeddings | Sentence Transformers |",
    "| Embedding Model | all-MiniLM-L6-v2 |",
    "| Vector Store | FAISS |",
    "| Document Processing | python-docx |",
    "| Numerical Processing | NumPy |",
    "| Development | Google Colab / VS Code |",
    "| Storage | Google Drive |",
    "",
    "---",
    "",
    "## 📁 Project Structure",
    "",
    "```text",
    "Enterprise_RAG_Assistant/",
    "│",
    "├── Data/",
    "│   └── documents/",
    "│       ├── Acceptable_Use_Policy.docx",
    "│       ├── Attendance_Policy.docx",
    "│       ├── Employee_Handbook.docx",
    "│       ├── IT_Security_Policy.docx",
    "│       ├── Leave_Policy.docx",
    "│       ├── Procurement_SOP.docx",
    "│       ├── Travel_Policy.docx",
    "│       └── Work_From_Home_Policy.docx",
    "│",
    "├── src/",
    "│   └── rag_engine.py",
    "│",
    "├── app/",
    "│   └── app.py",
    "│",
    "├── vectorstore/",
    "│   ├── tech_nova.index",
    "│   └── chunks.pkl",
    "│",
    "├── requirements.txt",
    "└── README.md",
    "```",
    "",
    "---",
    "",
    "## 📄 Knowledge Base",
    "",
    "The knowledge base currently contains 8 organizational documents:",
    "",
    "1. Acceptable Use Policy",
    "2. Attendance Policy",
    "3. Employee Handbook",
    "4. IT Security Policy",
    "5. Leave Policy",
    "6. Procurement SOP",
    "7. Travel Policy",
    "8. Work From Home Policy",
    "",
    "---",
    "",
    "## 🚀 Installation",
    "",
    "Clone the repository:",
    "",
    "```bash",
    "git clone <YOUR_GITHUB_REPOSITORY_URL>",
    "cd Enterprise_RAG_Assistant",
    "```",
    "",
    "Create a virtual environment:",
    "",
    "```bash",
    "python -m venv venv",
    "```",
    "",
    "Activate it on Windows:",
    "",
    "```bash",
    "venv\\Scripts\\activate",
    "```",
    "",
    "Install dependencies:",
    "",
    "```bash",
    "pip install -r requirements.txt",
    "```",
    "",
    "---",
    "",
    "## 🔑 Gemini API Configuration",
    "",
    "The application requires a Google Gemini API key.",
    "",
    "Windows PowerShell:",
    "",
    "```powershell",
    '$env:GEMINI_API_KEY="YOUR_API_KEY"',
    "```",
    "",
    "Linux / macOS:",
    "",
    "```bash",
    'export GEMINI_API_KEY="YOUR_API_KEY"',
    "```",
    "",
    "**Never commit the API key to GitHub.**",
    "",
    "For deployment platforms, configure the key using environment variables "
    "or secret management.",
    "",
    "---",
    "",
    "## ▶️ Run the Application",
    "",
    "From the project root:",
    "",
    "```bash",
    "streamlit run app/app.py",
    "```",
    "",
    "---",
    "",
    "## 💬 Example Questions",
    "",
    "### Leave Policy",
    "",
    "`How many annual leave days are provided?`",
    "",
    "### Work From Home",
    "",
    "`What is the work from home policy?`",
    "",
    "### Attendance",
    "",
    "`What are the attendance requirements?`",
    "",
    "### Travel",
    "",
    "`What is the company's travel policy?`",
    "",
    "### IT Security",
    "",
    "`What are the IT security requirements?`",
    "",
    "---",
    "",
    "## 🔎 Source Transparency",
    "",
    "Each generated answer can display its retrieved sources.",
    "",
    "The source information includes:",
    "",
    "- 📄 Document name",
    "- 🔹 Chunk ID",
    "- 📊 Relevance score",
    "- 📚 Retrieved passage count",
    "",
    "This provides traceability between the generated response and the "
    "internal company knowledge base.",
    "",
    "---",
    "",
    "## 🛡️ Grounded Answering",
    "",
    "The RAG prompt instructs the language model to answer using retrieved "
    "company documents rather than unrelated external knowledge.",
    "",
    "If relevant information cannot be found, the system can return:",
    "",
    "`This information is not available in the provided company documents.`",
    "",
    "This helps reduce unsupported answers and hallucinations.",
    "",
    "---",
    "",
    "## ⚙️ Current Configuration",
    "",
    "```text",
    "Embedding Model: all-MiniLM-L6-v2",
    "Embedding Dimension: 384",
    "Vector Store: FAISS",
    "LLM: gemini-3.5-flash-lite",
    "Relevance Threshold: 0.20",
    "```",
    "",
    "---",
    "",
    "## 🔮 Future Improvements",
    "",
    "- 🔐 Enterprise authentication",
    "- 👥 Role-based access control",
    "- 🗄️ Production vector database",
    "- 📊 Admin dashboard",
    "- 📝 PDF support",
    "- 📑 Additional document formats",
    "- 💾 Conversation persistence",
    "- 📈 RAG evaluation metrics",
    "- 🔍 Hybrid search",
    "- 🧠 Reranking models",
    "- ☁️ Cloud deployment",
    "- 🔄 Automatic document ingestion",
    "- 📌 Document version management",
    "- 🔒 Enterprise-level data security",
    "",
    "---",
    "",
    "## 📌 Project Status",
    "",
    "- ✅ Document ingestion",
    "- ✅ Embedding generation",
    "- ✅ FAISS vector search",
    "- ✅ RAG retrieval",
    "- ✅ Gemini generation",
    "- ✅ Source tracking",
    "- ✅ Document-specific filtering",
    "- ✅ Streamlit frontend",
    "- ✅ Document-specific pages",
    "- ✅ requirements.txt",
    "- 🔄 VS Code setup",
    "- 🔄 GitHub repository",
    "- 🔄 Deployment",
    "",
    "---",
    "",
    "## 📜 License",
    "",
    "This project is developed for educational and demonstration purposes.",
]

# ------------------------------------------------------------
# Write README
# ------------------------------------------------------------

README_PATH.write_text(
    "\n".join(readme_lines) + "\n",
    encoding="utf-8"
)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("=" * 70)
print("✅ README.md CREATED SUCCESSFULLY")
print("=" * 70)

print("\n📄 Location:")
print(README_PATH)

print("\n📊 README size:")
print(round(README_PATH.stat().st_size / 1024, 2), "KB")

print("\n📚 README lines:")
print(len(readme_lines))

print("\n✅ README is ready for GitHub")

✅ README.md CREATED SUCCESSFULLY

📄 Location:
/content/drive/MyDrive/Enterprise_RAG_Assistant/README.md

📊 README size:
6.85 KB

📚 README lines:
358

✅ README is ready for GitHub


In [83]:
# ============================================================
# STEP 11 — FINAL FILE CHECK
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant"
)

files_to_check = [
    PROJECT_ROOT / "app" / "app.py",
    PROJECT_ROOT / "src" / "rag_engine.py",
    PROJECT_ROOT / "requirements.txt",
    PROJECT_ROOT / "README.md",
    PROJECT_ROOT / "vectorstore" / "tech_nova.index",
    PROJECT_ROOT / "vectorstore" / "chunks.pkl",
]

print("=" * 70)
print("📦 FINAL PROJECT FILE CHECK")
print("=" * 70)

all_ok = True

for file_path in files_to_check:
    if file_path.exists():
        size_kb = file_path.stat().st_size / 1024
        print(f"✅ {file_path.relative_to(PROJECT_ROOT)} "
              f"({size_kb:.2f} KB)")
    else:
        print(f"❌ MISSING: {file_path.relative_to(PROJECT_ROOT)}")
        all_ok = False

print()
print("=" * 70)

if all_ok:
    print("🎉 ALL REQUIRED PROJECT FILES ARE PRESENT")
else:
    print("⚠️ SOME PROJECT FILES ARE MISSING")

print("=" * 70)

📦 FINAL PROJECT FILE CHECK
✅ app/app.py (33.53 KB)
✅ src/rag_engine.py (11.09 KB)
✅ requirements.txt (0.07 KB)
✅ README.md (6.85 KB)
✅ vectorstore/tech_nova.index (30.04 KB)
✅ vectorstore/chunks.pkl (10.86 KB)

🎉 ALL REQUIRED PROJECT FILES ARE PRESENT


In [44]:
# ============================================================
# FINAL PROJECT DIAGNOSTIC
# STEP 1 — CHECK CURRENT PROJECT WITHOUT MODIFYING ANYTHING
# ============================================================

from pathlib import Path
import sys
import inspect
import py_compile

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Enterprise_RAG_Assistant"
)

SRC_DIR = PROJECT_DIR / "src"
APP_DIR = PROJECT_DIR / "app"
VECTORSTORE_DIR = PROJECT_DIR / "vectorstore"
DOCS_DIR = PROJECT_DIR / "Data" / "documents"

RAG_FILE = SRC_DIR / "rag_engine.py"
APP_FILE = APP_DIR / "app.py"

print("=" * 80)
print("TECHNOVA ENTERPRISE RAG — FINAL PROJECT DIAGNOSTIC")
print("=" * 80)

# ------------------------------------------------------------
# 1. PROJECT DIRECTORIES
# ------------------------------------------------------------

print("\n1. PROJECT DIRECTORIES")
print("-" * 80)

print("Project:", PROJECT_DIR)
print("Project exists:", PROJECT_DIR.exists())

print("src exists:", SRC_DIR.exists())
print("app exists:", APP_DIR.exists())
print("vectorstore exists:", VECTORSTORE_DIR.exists())
print("documents exists:", DOCS_DIR.exists())


# ------------------------------------------------------------
# 2. IMPORTANT FILES
# ------------------------------------------------------------

print("\n2. IMPORTANT FILES")
print("-" * 80)

for file in [
    RAG_FILE,
    APP_FILE,
    VECTORSTORE_DIR / "tech_nova.index",
    VECTORSTORE_DIR / "chunks.pkl",
]:
    print(
        f"{'✅' if file.exists() else '❌'} "
        f"{file}"
    )


# ------------------------------------------------------------
# 3. DOCUMENT COUNT
# ------------------------------------------------------------

print("\n3. DOCUMENTS")
print("-" * 80)

if DOCS_DIR.exists():

    documents = sorted(
        DOCS_DIR.glob("*.docx")
    )

    print("DOCX documents:", len(documents))

    for document in documents:
        print("  📄", document.name)

else:

    print("❌ Documents directory not found.")


# ------------------------------------------------------------
# 4. RAG ENGINE SYNTAX
# ------------------------------------------------------------

print("\n4. RAG ENGINE SYNTAX")
print("-" * 80)

if RAG_FILE.exists():

    try:

        py_compile.compile(
            str(RAG_FILE),
            doraise=True
        )

        print("✅ rag_engine.py syntax is valid.")

    except py_compile.PyCompileError as e:

        print("❌ rag_engine.py has a syntax error.")
        print(e)

else:

    print("❌ rag_engine.py not found.")


# ------------------------------------------------------------
# 5. APP SYNTAX
# ------------------------------------------------------------

print("\n5. STREAMLIT APP SYNTAX")
print("-" * 80)

if APP_FILE.exists():

    try:

        py_compile.compile(
            str(APP_FILE),
            doraise=True
        )

        print("✅ app.py syntax is valid.")

    except py_compile.PyCompileError as e:

        print("❌ app.py has a syntax error.")
        print(e)

else:

    print("❌ app.py not found.")


# ------------------------------------------------------------
# 6. INSPECT RAG ENGINE
# ------------------------------------------------------------

print("\n6. CURRENT RAG ENGINE")
print("-" * 80)

if RAG_FILE.exists():

    rag_code = RAG_FILE.read_text(
        encoding="utf-8"
    )

    print("File size:", len(rag_code), "characters")

    checks = {
        "FAISS": "import faiss" in rag_code,
        "SentenceTransformer": "SentenceTransformer" in rag_code,
        "Gemini": "genai.Client" in rag_code,
        "retrieve()": "def retrieve" in rag_code,
        "document_name filter": "document_name=None" in rag_code,
        "build_rag_prompt()": "def build_rag_prompt" in rag_code,
        "generate_with_retry()": "def generate_with_retry" in rag_code,
        "answer_question()": "def answer_question" in rag_code,
        "relevance threshold": "RELEVANCE_THRESHOLD" in rag_code,
        "3.5 flash-lite": "gemini-3.5-flash-lite" in rag_code,
    }

    for name, found in checks.items():

        print(
            f"{'✅' if found else '❌'} {name}"
        )


# ------------------------------------------------------------
# 7. IMPORT CURRENT RAG ENGINE
# ------------------------------------------------------------

print("\n7. RAG ENGINE IMPORT")
print("-" * 80)

try:

    if str(PROJECT_DIR) not in sys.path:
        sys.path.insert(
            0,
            str(PROJECT_DIR)
        )

    import src.rag_engine as rag_engine

    print("✅ src.rag_engine imported.")

    print(
        "retrieve signature:",
        inspect.signature(
            rag_engine.retrieve
        )
    )

    print(
        "answer_question signature:",
        inspect.signature(
            rag_engine.answer_question
        )
    )

except Exception as e:

    print("❌ RAG engine import failed.")
    print(type(e).__name__, ":", e)


# ------------------------------------------------------------
# 8. VECTORSTORE
# ------------------------------------------------------------

print("\n8. VECTORSTORE")
print("-" * 80)

try:

    print(
        "FAISS vectors:",
        rag_engine.index.ntotal
    )

    print(
        "Embedding dimension:",
        rag_engine.index.d
    )

    print(
        "Chunks:",
        len(rag_engine.chunks)
    )

    print(
        "Relevance threshold:",
        getattr(
            rag_engine,
            "RELEVANCE_THRESHOLD",
            "NOT FOUND"
        )
    )

except Exception as e:

    print("❌ Could not inspect vectorstore.")
    print(type(e).__name__, ":", e)


# ------------------------------------------------------------
# 9. CURRENT RETRIEVAL TEST
# ------------------------------------------------------------

print("\n9. RETRIEVAL TEST")
print("-" * 80)

try:

    results = rag_engine.retrieve(
        "How many annual leave days do employees get?",
        top_k=3
    )

    print(
        "Retrieved results:",
        len(results)
    )

    for i, result in enumerate(
        results,
        start=1
    ):

        print(
            f"{i}. "
            f"{result['document']} "
            f"| Score: {result['score']:.4f}"
        )

    if results:
        print("✅ Retrieval is working.")
    else:
        print("⚠️ No results returned.")

except Exception as e:

    print("❌ Retrieval failed.")
    print(type(e).__name__, ":", e)


# ------------------------------------------------------------
# 10. DOCUMENT-SPECIFIC RETRIEVAL
# ------------------------------------------------------------

print("\n10. DOCUMENT-SPECIFIC RETRIEVAL")
print("-" * 80)

try:

    results = rag_engine.retrieve(
        "How many annual leave days do employees get?",
        top_k=3,
        document_name="Leave_Policy.docx"
    )

    print(
        "Leave Policy results:",
        len(results)
    )

    for result in results:

        print(
            f"  📄 {result['document']} "
            f"| Score: {result['score']:.4f}"
        )

    if results and all(
        r["document"] == "Leave_Policy.docx"
        for r in results
    ):

        print(
            "✅ Document-specific retrieval works."
        )

    else:

        print(
            "⚠️ Document-specific retrieval needs attention."
        )

except Exception as e:

    print("❌ Document-specific retrieval failed.")
    print(type(e).__name__, ":", e)


# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

print(
    "⚠️ No project files were modified by this diagnostic."
)

print(
    "Send me the complete output before running anything else."
)

print("=" * 80)

TECHNOVA ENTERPRISE RAG — FINAL PROJECT DIAGNOSTIC

1. PROJECT DIRECTORIES
--------------------------------------------------------------------------------
Project: /content/drive/MyDrive/Enterprise_RAG_Assistant
Project exists: True
src exists: True
app exists: True
vectorstore exists: True
documents exists: True

2. IMPORTANT FILES
--------------------------------------------------------------------------------
✅ /content/drive/MyDrive/Enterprise_RAG_Assistant/src/rag_engine.py
✅ /content/drive/MyDrive/Enterprise_RAG_Assistant/app/app.py
✅ /content/drive/MyDrive/Enterprise_RAG_Assistant/vectorstore/tech_nova.index
✅ /content/drive/MyDrive/Enterprise_RAG_Assistant/vectorstore/chunks.pkl

3. DOCUMENTS
--------------------------------------------------------------------------------
DOCX documents: 8
  📄 Acceptable_Use_Policy.docx
  📄 Attendance_Policy.docx
  📄 Employee_Handbook.docx
  📄 IT_Security_Policy.docx
  📄 Leave_Policy.docx
  📄 Procurement_SOP.docx
  📄 Travel_Policy.docx
  📄 Wor